In [2]:
#这个版本成功的使用了presistent但是没有搞好ms缓存，现在我要处理多场景问题了，在这里搞

In [ ]:
# ===================== ALL-IN-ONE: 3D simplex (tetrahedra) for PID-SP with gurobi_persistent =====================
# Requirements: pyomo, gurobi, numpy, scipy, plotly, tqdm, csv file "data.csv"
# -----------------------------------------------------------------------------------------

import numpy as np
import itertools as it
import csv
from tqdm import tqdm
import pyomo.environ as pyo
from pyomo.opt import SolverStatus, TerminationCondition
from pyomo.solvers.plugins.solvers.gurobi_persistent import GurobiPersistent
from scipy.spatial import Delaunay
import plotly.graph_objects as go
import matplotlib.pyplot as plt

# ===== debug bucket =====
LAST_DEBUG = None   # 如果 next_node == 顶点 或 撞车，会把当时的上下文塞进来

# ------------------------- Config knobs -------------------------
MIN_DIST   = 1e-8     # 去重阈值
ACTIVE_TOL = 1e-8     # active 判定容差
MS_AGG     = "sum"    # 单形 ms 聚合：'sum' 或 'mean'

# ------------------------- PID scenario model -------------------------
def build_pid_model(T=10, h=0.2, scen=None, weights=(1.0, 0.01),
                    bounds=None, use_cvar=False, alpha=0.95):
    assert scen is not None, "请提供一个场景字典"
    Ku, tau, d, sp = scen["Ku"], scen["tau"], scen["d"], scen["sp"]
    assert len(d) == T+1 and len(sp) == T+1

    if bounds is None:
        bounds = {}
    bx = bounds.get("x",  (-20, 20))
    bu = bounds.get("u",  (None, None))
    bKp= bounds.get("Kp", (0, 10))
    bKi= bounds.get("Ki", (0, 10))
    bKd= bounds.get("Kd", (0, 10))
    be = bounds.get("e",  (-100, 100))
    bI = bounds.get("I",  (-200, 200))

    m = pyo.ConcreteModel()
    m.T  = pyo.RangeSet(0, T)
    m.Tm = pyo.RangeSet(1, T)

    m.Kp = pyo.Var(bounds=bKp)
    m.Ki = pyo.Var(bounds=bKi)
    m.Kd = pyo.Var(bounds=bKd)

    m.x = pyo.Var(m.T, bounds=bx)
    m.u = pyo.Var(m.T, bounds=bu)
    m.e = pyo.Var(m.T, bounds=be)
    m.I = pyo.Var(m.T, bounds=bI)

    # error
    def _err_rule(m, t): return m.e[t] == sp[t] - m.x[t]
    m.err_def = pyo.Constraint(m.T, rule=_err_rule)

    # integral
    def _I_dyn(m, t): return m.I[t] == m.I[t-1] + h*m.e[t]
    m.I_dyn = pyo.Constraint(m.Tm, rule=_I_dyn)

    # plant
    def _x_dyn(m, t):
        return m.x[t] == m.x[t-1] + (h/tau)*(-m.x[t] + Ku*m.u[t] + d[t])
    m.x_dyn = pyo.Constraint(m.Tm, rule=_x_dyn)

    # pid
    def _pid_rule(m, t):
        if t == 0:
            return m.u[t] == m.Kp*m.e[t] + m.Ki*m.I[t]
        return m.u[t] == m.Kp*m.e[t] + m.Ki*m.I[t] + m.Kd*(m.e[t]-m.e[t-1])/h
    m.pid = pyo.Constraint(m.T, rule=_pid_rule)

    m.x0 = pyo.Constraint(expr=m.x[0] == 0)
    m.I0 = pyo.Constraint(expr=m.I[0] == 0)

    w_e, w_u = weights
    m.cost = pyo.Expression(expr=sum(h*(w_e*m.e[t]**2 + w_u*m.u[t]**2) for t in m.T))
    m.obj_expr = pyo.Expression(expr=m.cost)  # 只保留表达式，目标在持久化阶段统一创建
    return m, [m.Kp, m.Ki, m.Kd]

def load_scenarios_from_csv(csv_path: str, T: int | None = None,
                            sp0: float = 0.0, sp1: float = 0.5,
                            ku_col: str = "tau_us", tau_col: str = "tau_xs",
                            disturb_prefix: str = "disturbance_",
                            setpoint_change_col: str = "setpoint_change"):
    scens = []
    # 推断 T
    if T is None:
        with open(csv_path, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            fields  = reader.fieldnames or []
            max_idx = -1
            for name in fields:
                if name.startswith(disturb_prefix):
                    try:
                        k = int(name[len(disturb_prefix):])
                        max_idx = max(max_idx, k)
                    except:
                        pass
            if max_idx < 0:
                raise ValueError(f"未找到扰动列前缀 {disturb_prefix}k")
            T = max_idx

    with open(csv_path, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            Ku  = float(row[ku_col])
            tau = float(row[tau_col])

            d = []
            for t in range(T+1):
                col = f"{disturb_prefix}{t}"
                d.append(float(row[col]))

            sp = [sp1]*(T+1)
            if setpoint_change_col in row and row[setpoint_change_col] != "":
                try:
                    t_star = int(float(row[setpoint_change_col]))
                    for t in range(T+1):
                        sp[t] = sp0 if t < t_star else sp1
                except:
                    pass

            scens.append({"Ku": Ku, "tau": tau, "d": d, "sp": sp})
    return scens, T

def build_models_from_csv(csv_path: str, h: float = 0.2,
                          weights=(1.0, 0.01), bounds=None,
                          sp0: float = 0.0, sp1: float = 0.5,
                          ku_col: str = "tau_us", tau_col: str = "tau_xs",
                          disturb_prefix: str = "disturbance_",
                          setpoint_change_col: str = "setpoint_change",
                          max_scenarios=None, skip=0):
    scens, T = load_scenarios_from_csv(
        csv_path=csv_path, T=None, sp0=sp0, sp1=sp1,
        ku_col=ku_col, tau_col=tau_col,
        disturb_prefix=disturb_prefix,           # ✅ FIX 这里：用 disturb_prefix（参数），不是 disturbance_
        setpoint_change_col=setpoint_change_col,
    )
    if skip or max_scenarios:
        scens = scens[skip: (skip + max_scenarios) if max_scenarios else None]

    model_list, first_stg_vars_list = [], []
    for scen in scens:
        m, yvars = build_pid_model(T=T, h=h, scen=scen, weights=weights, bounds=bounds)
        model_list.append(m)
        first_stg_vars_list.append(yvars)

    m_tmpl_list = [model_list[0], first_stg_vars_list[0]]
    return model_list, first_stg_vars_list, m_tmpl_list, T


# ------------------------- Persistent wrappers -------------------------
class BaseBundle:
    """每个场景的基础模型（计算真实Q）+ 持久化求解器"""
    def __init__(self, model: pyo.ConcreteModel, options: dict | None = None):
        self.model = model
        self.gp = GurobiPersistent()
        self.gp.set_instance(model)
        if hasattr(model, 'obj'):
            model.del_component('obj')
        model.obj = pyo.Objective(expr=model.obj_expr, sense=pyo.minimize)
        self.gp.set_objective(model.obj)
        if options:
            self.gp.set_gurobi_param('MIPGap', options.get('MIPGap', 1e-1))
            self.gp.set_gurobi_param('NumericFocus', options.get('NumericFocus', 1))
            self.gp.set_gurobi_param('Presolve', options.get('Presolve', 2))
            self.gp.set_gurobi_param('NonConvex', options.get('NonConvex', 2))
            if 'TimeLimit' in options:
                self.gp.set_gurobi_param('TimeLimit', options['TimeLimit'])

    def eval_at(self, first_vars, first_vals):
        for v, val in zip(first_vars, first_vals):
            v.fix(float(val))
            self.gp.update_var(v)
        self.gp.solve(load_solutions=True)
        val = float(pyo.value(self.model.obj_expr))
        for v in first_vars:
            v.unfix()
            self.gp.update_var(v)
        return val
class MSBundle:
    """ms 模板（持久化）：每次更新四面体时 remove+add 约束，兼容没有
    set_linear_coefficient / set_constraint_expr 的 Pyomo 版本。
    """
    def __init__(self, model_base: pyo.ConcreteModel, first_vars, options: dict | None = None):
        # 克隆该场景模型
        m = model_base.clone()

        # 重心变量与和=1
        m.lam_index = pyo.RangeSet(0, 3)
        m.lam = pyo.Var(m.lam_index, domain=pyo.NonNegativeReals)
        m.lam_sum = pyo.Constraint(expr=sum(m.lam[j] for j in m.lam_index) == 1.0)

        # 一阶段变量引用
        self.Kp = m.find_component(first_vars[0].name)
        self.Ki = m.find_component(first_vars[1].name)
        self.Kd = m.find_component(first_vars[2].name)
        if any(v is None for v in (self.Kp, self.Ki, self.Kd)):
            raise RuntimeError("克隆模型中找不到 Kp/Ki/Kd")

        # 先放占位约束（系数0），真正的表达式在 update_tetra 里重建
        m.link_kp = pyo.Constraint(expr=self.Kp == sum(0.0 * m.lam[j] for j in m.lam_index))
        m.link_ki = pyo.Constraint(expr=self.Ki == sum(0.0 * m.lam[j] for j in m.lam_index))
        m.link_kd = pyo.Constraint(expr=self.Kd == sum(0.0 * m.lam[j] for j in m.lam_index))

        m.As = pyo.Var()
        m.As_def = pyo.Constraint(expr=m.As == sum(0.0 * m.lam[j] for j in m.lam_index))

        # 目标
        if hasattr(m, 'obj'):
            m.del_component('obj')
        m.obj = pyo.Objective(expr=m.obj_expr - m.As, sense=pyo.minimize)

        # 持久化求解器
        self.model = m
        self.gp = GurobiPersistent()
        self.gp.set_instance(m)
        self.gp.set_objective(m.obj)
        if options:
            self.gp.set_gurobi_param('MIPGap', options.get('MIPGap', 1e-1))
            self.gp.set_gurobi_param('NumericFocus', options.get('NumericFocus', 1))
            self.gp.set_gurobi_param('Presolve', options.get('Presolve', 2))
            self.gp.set_gurobi_param('NonConvex', options.get('NonConvex', 2))
            if 'TimeLimit' in options:
                self.gp.set_gurobi_param('TimeLimit', options['TimeLimit'])

        # 缓存/别名
        self.lam = m.lam
        self.link_kp = m.link_kp
        self.link_ki = m.link_ki
        self.link_kd = m.link_kd
        self.As     = m.As
        self.As_def = m.As_def
        self._V_cached = None  # [(x,y,z)]*4

    # ---- 工具：构建表达式 & 热替换约束（先从求解器移除，再从模型删掉，重建并 add 回去） ----
    def _expr_link(self, lhs_var, coeffs):
        return lhs_var == sum(float(coeffs[j]) * self.lam[j] for j in range(4))

    def _replace_constraint(self, attr_name: str, expr):
        # 1) 从求解器里移除旧约束（若已存在）
        old_con = getattr(self.model, attr_name)
        try:
            self.gp.remove_constraint(old_con)
        except Exception:
            pass  # 第一次可能还没被单独 add 过

        # 2) 从模型里删除旧组件
        self.model.del_component(old_con)

        # 3) 以相同名字重建新约束并加入模型
        new_con = pyo.Constraint(expr=expr)
        self.model.add_component(attr_name, new_con)

        # 4) 通知持久化求解器新增该约束
        self.gp.add_constraint(getattr(self.model, attr_name))

    # ---- 外部接口：写入四面体数据并求解 ----
    def update_tetra(self, tet_vertices, fverts_scene):
        # 稳定排序（与外层一致）
        pairs = sorted(
            [(tuple(map(float, tet_vertices[j])), float(fverts_scene[j])) for j in range(4)],
            key=lambda kv: (kv[0][0], kv[0][1], kv[0][2])
        )
        V = [kv[0] for kv in pairs]  # 顶点坐标
        F = [kv[1] for kv in pairs]  # 顶点值
        self._V_cached = V

        vx = [V[j][0] for j in range(4)]
        vy = [V[j][1] for j in range(4)]
        vz = [V[j][2] for j in range(4)]

        # 用 remove+add 的方式重建四条等式
        self._replace_constraint('link_kp', self._expr_link(self.Kp, vx))
        self._replace_constraint('link_ki', self._expr_link(self.Ki, vy))
        self._replace_constraint('link_kd', self._expr_link(self.Kd, vz))
        self._replace_constraint('As_def',  self._expr_link(self.As, F))

    def solve(self):
        res = self.gp.solve(load_solutions=True)
        ok = (res.solver.status == SolverStatus.ok) and \
             (res.solver.termination_condition in {
                 TerminationCondition.optimal,
                 TerminationCondition.locallyOptimal
             })
        return ok

    def get_ms_and_point(self):
        ms_val = float(pyo.value(self.model.obj))
        lam_star = np.array([pyo.value(self.lam[j]) for j in range(4)], dtype=float)
        V = np.array(self._V_cached, dtype=float)
        new_pt = lam_star @ V
        return ms_val, lam_star, tuple(map(float, new_pt))

class AggregateMSBundle:
    """
    多场景共享一组 λ 的 ms 子问题：
      minimize  Σ_ω [ obj_expr_ω(K(λ)) - Σ_j λ_j f_ω(v_j) ]
      s.t.      λ >= 0, Σ λ = 1
                K(λ) = Σ_j λ_j v_j  （对全部场景共用同一组 λ，但各场景各自的 Kp/Ki/Kd 由各自模型承载）
    """
    def __init__(self, model_bases: list[pyo.ConcreteModel],
                 first_vars_list: list[list[pyo.Var]],
                 options: dict | None = None):
        S = len(model_bases)
        assert S == len(first_vars_list) and S >= 1, "场景数不一致或为0"

        # 顶层模型
        M = pyo.ConcreteModel()

        # 共享重心变量
        M.lam_index = pyo.RangeSet(0, 3)
        M.lam = pyo.Var(M.lam_index, domain=pyo.NonNegativeReals)
        M.lam_sum = pyo.Constraint(expr=sum(M.lam[j] for j in M.lam_index) == 1.0)

        # 把每个场景的模型 clone 进来，作为 Block
        M.sc = pyo.Block(range(S))
        self.blocks = []
        self.Kp = {}
        self.Ki = {}
        self.Kd = {}
        self.link_kp = {}
        self.link_ki = {}
        self.link_kd = {}
        self.As = {}
        self.As_def = {}

        # 目标：Σ obj_expr_ω - Σ As_ω
        obj_terms = []
        as_terms  = []

        for w in range(S):
            mw = model_bases[w].clone()
            M.sc[w].transfer_attributes_from(mw)  # 把场景模型的组件直接放进 Block
            self.blocks.append(M.sc[w])

            # 一阶段变量引用
            Kp_w = M.sc[w].find_component(first_vars_list[w][0].name)
            Ki_w = M.sc[w].find_component(first_vars_list[w][1].name)
            Kd_w = M.sc[w].find_component(first_vars_list[w][2].name)
            if any(v is None for v in (Kp_w, Ki_w, Kd_w)):
                raise RuntimeError(f"场景 {w} 中找不到 Kp/Ki/Kd")

            # 先用0系数的占位式；真正的表达式在 update_tetra() 热替换
            M.add_component(f"link_kp_{w}", pyo.Constraint(expr=Kp_w == sum(0.0 * M.lam[j] for j in M.lam_index)))
            M.add_component(f"link_ki_{w}", pyo.Constraint(expr=Ki_w == sum(0.0 * M.lam[j] for j in M.lam_index)))
            M.add_component(f"link_kd_{w}", pyo.Constraint(expr=Kd_w == sum(0.0 * M.lam[j] for j in M.lam_index)))

            M.add_component(f"As_{w}", pyo.Var())
            M.add_component(f"As_def_{w}", pyo.Constraint(expr=getattr(M, f"As_{w}") == sum(0.0 * M.lam[j] for j in M.lam_index)))

            # 缓存句柄
            self.Kp[w], self.Ki[w], self.Kd[w] = Kp_w, Ki_w, Kd_w
            self.link_kp[w] = getattr(M, f"link_kp_{w}")
            self.link_ki[w] = getattr(M, f"link_ki_{w}")
            self.link_kd[w] = getattr(M, f"link_kd_{w}")
            self.As[w]      = getattr(M, f"As_{w}")
            self.As_def[w]  = getattr(M, f"As_def_{w}")

            obj_terms.append(M.sc[w].obj_expr)
            as_terms.append(self.As[w])

        # 统一目标（用 sum；若 MS_AGG='mean' 也不影响最优 λ，仅缩放）
        if hasattr(M, 'obj'):
            M.del_component('obj')
        M.obj = pyo.Objective(expr=sum(obj_terms) - sum(as_terms), sense=pyo.minimize)

        # 持久化绑定
        self.model = M
        self.lam = M.lam
        self.gp = GurobiPersistent()
        self.gp.set_instance(M)
        self.gp.set_objective(M.obj)
        if options:
            self.gp.set_gurobi_param('MIPGap', options.get('MIPGap', 1e-1))
            self.gp.set_gurobi_param('NumericFocus', options.get('NumericFocus', 1))
            self.gp.set_gurobi_param('Presolve', options.get('Presolve', 2))
            self.gp.set_gurobi_param('NonConvex', options.get('NonConvex', 2))
            if 'TimeLimit' in options:
                self.gp.set_gurobi_param('TimeLimit', options['TimeLimit'])

        # 顶点缓存（用于返回新点坐标）
        self._V_cached = None
        self.S = S

    # ---- 内部：构建表达式 / 热替换（remove + add） ----
    def _expr_link(self, lhs_var, coeffs):
        return lhs_var == sum(float(coeffs[j]) * self.lam[j] for j in range(4))

    def _replace_constraint(self, con_name: str, expr):
        # 从求解器移除旧约束（若存在）
        old_con = getattr(self.model, con_name)
        try:
            self.gp.remove_constraint(old_con)
        except Exception:
            pass
        # 从模型中删除
        self.model.del_component(old_con)
        # 以相同名字重建并加入模型+求解器
        new_con = pyo.Constraint(expr=expr)
        self.model.add_component(con_name, new_con)
        self.gp.add_constraint(getattr(self.model, con_name))

    # ---- 外部接口 ----
    def update_tetra(self, tet_vertices, fverts_per_scene):
        """
        tet_vertices: [(x,y,z)] * 4
        fverts_per_scene: 长度 S 的列表；每项是该场景的四顶点值 [f_ω(v0),...,f_ω(v3)]
        """
        # 统一排序（按顶点坐标），并对所有场景的 fverts 应用同一排序
        pairs = sorted([(tuple(map(float, tet_vertices[j])), j) for j in range(4)],
                       key=lambda kv: (kv[0][0], kv[0][1], kv[0][2]))
        order = [orig_j for (_, orig_j) in pairs]
        V = [tet_vertices[j] for j in order]
        self._V_cached = [tuple(map(float, v)) for v in V]

        vx = [V[j][0] for j in range(4)]
        vy = [V[j][1] for j in range(4)]
        vz = [V[j][2] for j in range(4)]

        # 对每个场景，重建 link_kp/ki/kd 与 As_def
        for w in range(self.S):
            Fw = [float(fverts_per_scene[w][j]) for j in order]
            self._replace_constraint(f"link_kp_{w}", self._expr_link(self.Kp[w], vx))
            self._replace_constraint(f"link_ki_{w}", self._expr_link(self.Ki[w], vy))
            self._replace_constraint(f"link_kd_{w}", self._expr_link(self.Kd[w], vz))
            self._replace_constraint(f"As_def_{w}",  self._expr_link(self.As[w], Fw))

    def solve(self):
        res = self.gp.solve(load_solutions=True)
        ok = (res.solver.status == SolverStatus.ok) and \
             (res.solver.termination_condition in {
                 TerminationCondition.optimal,
                 TerminationCondition.locallyOptimal
             })
        return ok

    def get_ms_and_point(self):
        # 目标值 = Σ_ω F_ω(λ*)，即“sum 聚合”的 ms_total
        ms_sum = float(pyo.value(self.model.obj))
        ms_total = ms_sum if (MS_AGG == "sum") else (ms_sum / float(self.S))
        lam_star = np.array([pyo.value(self.lam[j]) for j in range(4)], dtype=float)
        V = np.array(self._V_cached, dtype=float)
        new_pt = lam_star @ V
        return ms_total, lam_star, tuple(map(float, new_pt))


def build_persistent_bundles(model_list, first_stg_vars_list, gurobi_options: dict):
    base_bundles = []
    ms_bundles   = []
    for m, yvars in zip(model_list, first_stg_vars_list):
        base_bundles.append(BaseBundle(m, gurobi_options))
        ms_bundles.append(MSBundle(m, yvars, gurobi_options))
    return base_bundles, ms_bundles

# ------------------------- Basic utils -------------------------
def corners_from_var_bounds(vars_3):
    bnds = []
    for v in vars_3:
        lb, ub = v.lb, v.ub
        if lb is None or ub is None:
            raise ValueError(f"{v.name} 缺少上下界")
        bnds.append((float(lb), float(ub)))
    return [tuple(p) for p in it.product(*[(lo, hi) for (lo,hi) in bnds])]

def too_close(p, nodes, tol=MIN_DIST):
    return any(np.linalg.norm(np.asarray(p)-np.asarray(q)) < tol for q in nodes)

def evaluate_Q_at(base_bundle: BaseBundle, first_stg_vars, first_stg_vals):
    return base_bundle.eval_at(first_stg_vars, first_stg_vals)

def tet_volume(verts):
    V = np.array(verts, float)
    v0, v1, v2, v3 = V
    return float(abs(np.linalg.det(np.stack([v1 - v0, v2 - v0, v3 - v0], axis=1))) / 6.0)

def tet_quality(verts):
    V = np.array(verts, float)
    edges = [np.linalg.norm(V[i] - V[j]) for (i, j) in it.combinations(range(4), 2)]
    denom = float(np.sum(np.power(edges, 3))) + 1e-16
    vol = tet_volume(verts)
    return float(6.0 * vol / denom)

# ------------------------- Single tetra & scene: ms solve (persistent) -------------------------
def ms_on_tetra_for_scene(ms_bundle: MSBundle, tet_vertices, fverts_scene):
    ms_bundle.update_tetra(tet_vertices, fverts_scene)
    ok = ms_bundle.solve()
    if not ok:
        return float('inf'), None, None
    ms_val, lam_star, new_pt = ms_bundle.get_ms_and_point()
    return ms_val, lam_star, new_pt

# ------------------------- Evaluate all tetrahedra -------------------------
def evaluate_all_tetra(nodes, scen_values, ms_bundles, first_vars_list):
    pts = np.asarray(nodes, dtype=float)
    if len(pts) < 4:
        return None, []
    tri = Delaunay(pts)
    S = len(ms_bundles)

    mins = pts.min(axis=0)
    maxs = pts.max(axis=0)
    diam = float(np.linalg.norm(maxs - mins))
    vol_tol = 1e-12 * max(diam**3, 1.0)

    per_tet = []
    for k, simp in enumerate(tri.simplices):
        idxs = list(map(int, simp))
        verts = [tuple(pts[i]) for i in idxs]

        v0, v1, v2, v3 = np.array(verts)
        vol = abs(np.linalg.det(np.stack([v1 - v0, v2 - v0, v3 - v0], axis=1))) / 6.0
        if vol < vol_tol:
            continue

        fverts_per_scene = [[scen_values[ω][i] for i in idxs] for ω in range(S)]
        fverts_sum = [sum(fverts_per_scene[ω][j] for ω in range(S)) for j in range(4)]

        ms_scene = []
        xms_scene = []
        for ω in range(S):
            ms_val, lam_star, new_pt = ms_on_tetra_for_scene(
                ms_bundles[ω], verts, fverts_per_scene[ω]
            )
            ms_scene.append(ms_val)
            xms_scene.append(new_pt)

        if MS_AGG == "sum":
            ms_total = float(np.sum(ms_scene))
        elif MS_AGG == "mean":
            ms_total = float(np.mean(ms_scene))
        else:
            raise ValueError("MS_AGG must be 'sum' or 'mean'")

        LB = float(np.min(fverts_sum) + ms_total)
        UB = float(np.max(fverts_sum) + ms_total)

        best_scene = int(np.argmin(ms_scene))
        x_ms_best = xms_scene[best_scene]

        per_tet.append({
            "simplex_index": k,
            "vert_idx": idxs,
            "verts": verts,
            "fverts_sum": fverts_sum,
            "ms_per_scene": ms_scene,
            "ms": ms_total,
            "LB": LB,
            "UB": UB,
            "x_ms_best_scene": x_ms_best,
            "best_scene": best_scene,
            "volume": vol,
        })

    return tri, per_tet

def evaluate_all_tetra_agg(nodes, scen_values, aggregate_bundle, first_vars_list):
    """
    使用“共享 λ”的聚合 ms：
      - 对每个四面体：把所有场景的四顶点值打包给 AggregateMSBundle
      - 返回 per_tet，其中：
          'ms' 是聚合后的 ms_total（sum 或 mean，取决于 MS_AGG）
          'x_ms_best_scene' 用共享 λ 得到的落点（不再取单场景最优）
      - 不再返回 'ms_per_scene'（若要打印 per-scenario，可另外保留单场景 MSBundle 计算，但不建议）
    """
    pts = np.asarray(nodes, dtype=float)
    if len(pts) < 4:
        return None, []
    tri = Delaunay(pts)
    S = len(scen_values)

    # 尺度基准/体积阈值
    mins = pts.min(axis=0)
    maxs = pts.max(axis=0)
    diam = float(np.linalg.norm(maxs - mins))
    vol_tol = 1e-12 * max(diam**3, 1.0)

    per_tet = []
    for k, simp in enumerate(tri.simplices):
        idxs = list(map(int, simp))
        verts = [tuple(pts[i]) for i in idxs]

        # 体积，过滤劣形
        v0, v1, v2, v3 = np.array(verts)
        vol = abs(np.linalg.det(np.stack([v1 - v0, v2 - v0, v3 - v0], axis=1))) / 6.0
        if vol < vol_tol:
            continue

        # 各场景四顶点值（按 idxs）
        fverts_per_scene = [[scen_values[w][i] for i in idxs] for w in range(S)]
        # 四顶点的“总 f”（用于 UB/LB ）
        fverts_sum = [sum(fverts_per_scene[w][j] for w in range(S)) for j in range(4)]

        # 共享 λ 的聚合 ms
        aggregate_bundle.update_tetra(verts, fverts_per_scene)
        ok = aggregate_bundle.solve()
        if not ok:
            # 若失败，给个保守值：跳过/或设置很大
            ms_total = float('inf')
            x_ms = None
        else:
            ms_total, lam_star, x_ms = aggregate_bundle.get_ms_and_point()

        # LB/UB（聚合一致）
        LB = float(np.min(fverts_sum) + (ms_total if MS_AGG == "sum" else ms_total * S)) if MS_AGG == "mean" else float(np.min(fverts_sum) + ms_total)
        UB = float(np.max(fverts_sum) + (ms_total if MS_AGG == "sum" else ms_total * S)) if MS_AGG == "mean" else float(np.max(fverts_sum) + ms_total)
        # 上面两行写得绕，是为了兼容你之前的“sum 的 fverts_sum + ms_total”。若 MS_AGG='mean'，
        # 你的 fverts_sum 是 sum(场景) 而不是均值，因此把 ms_total*S 再对齐到“sum 标尺”。

        per_tet.append({
            "simplex_index": k,
            "vert_idx": idxs,
            "verts": verts,
            "fverts_sum": fverts_sum,   # sum 标尺
            "ms": ms_total,             # 若 MS_AGG='mean'，这里是 mean；若 'sum'，就是 sum
            "LB": LB,
            "UB": UB,
            "x_ms_best_scene": x_ms,    # 现在是“共享 λ”的全局落点
            "best_scene": -1,           # 不再有单场景最优
            "volume": vol,
        })

    return tri, per_tet


# ------------------------- Pretty print -------------------------
def print_tetra_table(per_tet, active_mask, purple_set=None, prec=6):
    purple_set = set() if purple_set is None else set(purple_set)
    per_tet = sorted(per_tet, key=lambda r: r["simplex_index"])
    tet_ids = [r["simplex_index"] for r in per_tet]
    active_set = {tid for tid in tet_ids if active_mask.get(tid, False)}

    def _mark(tid):
        s = f"T{tid}"
        flags = []
        if tid in active_set:  flags.append("*")
        if tid in purple_set:  flags.append("^")
        return s + ("".join(flags) if flags else "")

    header = ["row\\simp"] + [_mark(tid) for tid in tet_ids]
    rows = [
        ["UB"] + [f"{r['UB']:.{prec}f}" for r in per_tet],
        ["LB"] + [f"{r['LB']:.{prec}f}" for r in per_tet],
        ["ms"] + [f"{r['ms']:.3e}"       for r in per_tet],
    ]
    table = [header] + rows
    colw = [max(len(str(row[c])) for row in table) + 2 for c in range(len(header))]

    RED, PURPLE, RESET = "\033[31m", "\033[35m", "\033[0m"
    def colorize(col_idx, s):
        if col_idx == 0:
            return s
        tid = tet_ids[col_idx-1]
        if tid in purple_set:
            return f"{PURPLE}{s}{RESET}"
        elif tid in active_set:
            return f"{RED}{s}{RESET}"
        return s

    print("\n== Per-tetra summary ==")
    print("".join(colorize(c, str(header[c]).ljust(colw[c])) for c in range(len(header))))
    print("-"*sum(colw))
    for r in rows:
        line = []
        for c in range(len(header)):
            cell = str(r[c])
            pad  = cell.ljust(colw[c]) if c==0 else cell.rjust(colw[c])
            line.append(colorize(c, pad))
        print("".join(line))
    print("(红色列=active；紫色列=包含当前最小节点的单形；第1行=UB，第2行=LB，第3行=ms)\n")

def min_dist_to_nodes(pt, nodes):
    P = np.asarray(pt, float)
    X = np.asarray(nodes, float)
    return float(np.min(np.linalg.norm(X - P, axis=1)))

def print_per_scenario_ms(per_tet, max_scenarios_to_print=10, prec=3):
    per_tet = sorted(per_tet, key=lambda r: r["simplex_index"])
    if not per_tet or "ms_per_scene" not in per_tet[0]:
        return
    S = len(per_tet[0]["ms_per_scene"])
    show = min(S, max_scenarios_to_print)
    head = "simp | " + " ".join([f"s{j}".rjust(10) for j in range(show)])
    print("== Per-tetra per-scenario ms (showing first", show, "of", S, "scenes) ==")
    print(head); print("-"*len(head))
    for r in per_tet:
        arr = r["ms_per_scene"][:show]
        sline = " ".join([f"{v:.{prec}e}".rjust(10) for v in arr])
        print(f"{r['simplex_index']:>4d} | {sline}")
    if show < S:
        print(f"... ({S-show} scenes omitted)")
    print()

# ------------------------- Plotly visualization -------------------------
def plot_iteration_plotly(iter_id, nodes, tri, active_mask, ub_node, next_node, per_tet,
                          highlight_simplices=None):
    import numpy as np
    import plotly.graph_objects as go

    if highlight_simplices is None:
        highlight_simplices = set()
    else:
        highlight_simplices = set(highlight_simplices)

    fig = go.Figure()
    nodes = np.asarray(nodes, float)

    if len(nodes) > 0:
        fig.add_trace(go.Scatter3d(
            x=nodes[:, 0], y=nodes[:, 1], z=nodes[:, 2],
            mode='markers',
            marker=dict(size=4, color="black"),
            name='nodes'
        ))

    if ub_node is not None:
        fig.add_trace(go.Scatter3d(
            x=[ub_node[0]], y=[ub_node[1]], z=[ub_node[2]],
            mode='markers',
            marker=dict(size=7, symbol="circle", color="green"),
            name='current min node'
        ))

    if next_node is not None:
        fig.add_trace(go.Scatter3d(
            x=[next_node[0]], y=[next_node[1]], z=[next_node[2]],
            mode='markers',
            marker=dict(size=8, symbol="diamond", color="#1976d2"),
            name='next node'
        ))

    def _is_same_point(a, b, atol=1e-6):
        if a is None or b is None:
            return False
        return np.linalg.norm(np.asarray(a, float) - np.asarray(b, float)) <= float(atol)

    if tri is not None:
        legend_mesh_added = False
        legend_edge_added = False

        for r in per_tet:
            sid = r["simplex_index"]
            if not active_mask.get(sid, False):
                continue

            verts = np.array(r["verts"], dtype=float)
            highlight_by_next = _is_same_point(next_node, r.get("x_ms_best_scene", None), atol=1e-6)
            highlight = highlight_by_next or (sid in highlight_simplices)

            mesh_color = "#ff5722" if highlight else "#ffb74d"
            edge_color = "darkorange"
            edge_width = 4 if highlight else 3
            mesh_opacity = 0.45 if highlight else 0.35

            I = [0, 0, 0, 1]
            J = [1, 1, 2, 2]
            K = [2, 3, 3, 3]

            fig.add_trace(go.Mesh3d(
                x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
                i=I, j=J, k=K,
                color=mesh_color,
                opacity=mesh_opacity,
                showscale=False,
                name="active simplex",
                showlegend=(not legend_mesh_added)
            ))
            legend_mesh_added = True

            edges = [(0,1), (0,2), (0,3), (1,2), (1,3), (2,3)]
            for (a, b) in edges:
                pa, pb = verts[a], verts[b]
                fig.add_trace(go.Scatter3d(
                    x=[pa[0], pb[0]],
                    y=[pa[1], pb[1]],
                    z=[pa[2], pb[2]],
                    mode='lines',
                    line=dict(width=edge_width, color=edge_color),
                    name='active edge',
                    showlegend=(not legend_edge_added)
                ))
            legend_edge_added = True

            cx, cy, cz = np.mean(verts, axis=0)
            qtxt = ""
            if "quality" in r and r["quality"] is not None:
                try:
                    qtxt = f"<br>q={float(r['quality']):.3e}"
                except Exception:
                    qtxt = ""
            txt = (f"simp={sid}"
                   f"<br>LB={float(r['LB']):.6f}"
                   f"<br>UB={float(r['UB']):.6f}"
                   f"<br>ms={float(r['ms']):.3e}"
                   f"<br>vol={float(r['volume']):.3e}"
                   f"{qtxt}")

            fig.add_trace(go.Scatter3d(
                x=[cx], y=[cy], z=[cz],
                mode='markers',
                marker=dict(size=1, opacity=0.0),
                text=[txt], hoverinfo="text",
                name="tetra info",
                showlegend=False
            ))

    fig.update_layout(
        title=f"Iteration {iter_id}",
        scene=dict(
            xaxis_title="Kp",
            yaxis_title="Ki",
            zaxis_title="Kd",
            aspectmode="cube",
            zaxis=dict(tickformat=".2f"),
        ),
        width=980,
        height=720,
        legend=dict(itemsizing="constant")
    )
    fig.update_traces(
        hovertemplate="x: %{x:.6f}<br>y: %{y:.6f}<br>z: %{z:.6f}",
        selector=dict(type='scatter3d')
    )
    fig.show()

# ------------------------- MAIN LOOP -------------------------
def run_pid_simplex_3d(base_bundles, ms_bundles, model_list, first_vars_list,
                       target_nodes=30, min_dist=MIN_DIST, active_tol=ACTIVE_TOL, verbose=True,
                       agg_bundle=None):
    global LAST_DEBUG
    LB_hist, UB_hist, ms_hist, node_count = [], [], [], []
    UB_node_hist, add_node_hist = [], []
    ms_a_hist, ms_b_hist = [], []
    active_ratio_hist = []

    S = len(model_list)
    nodes = corners_from_var_bounds(first_vars_list[0])

    bounds_arr = np.array([[float(v.lb), float(v.ub)] for v in first_vars_list[0]], float)
    diam = float(np.linalg.norm(bounds_arr[:,1] - bounds_arr[:,0]))
    min_dist = float(min_dist)

    # 缓存 f_ω(node_i)
    scen_values = [[None]*len(nodes) for _ in range(S)]
    for i, node in enumerate(nodes):
        for ω in range(S):
            scen_values[ω][i] = evaluate_Q_at(base_bundles[ω], first_vars_list[ω], node)

    it = 0
    stop_due_to_collision = False
    while len(nodes) < target_nodes:
        # 1) 全局 UB
        f_sum_per_node = [
            sum(scen_values[ω][i] for ω in range(S))
            for i in range(len(nodes))
        ]
        ub_idx = int(np.argmin(f_sum_per_node))
        UB_global = float(f_sum_per_node[ub_idx])
        UB_node = tuple(nodes[ub_idx])

        # 2) 评估所有四面体（使用持久化 ms）
        if agg_bundle is not None:
            tri, per_tet = evaluate_all_tetra_agg(nodes, scen_values, agg_bundle, first_vars_list)
        else:
            tri, per_tet = evaluate_all_tetra(nodes, scen_values, ms_bundles, first_vars_list)

        if tri is None or not per_tet:
            if verbose:
                print("Not enough nodes to make tetrahedra; stop.")
            break

        # 3) active mask（按 UB 过滤 + 形状质量）
        active_mask = {
            r["simplex_index"]: (r["LB"] <= UB_global + active_tol)
            for r in per_tet
        }
        q_cut = 1e-3
        for r in per_tet:
            sid = r["simplex_index"]
            if not active_mask.get(sid, False):
                continue
            q = tet_quality(r["verts"])
            if q < q_cut:
                active_mask[sid] = False

        # 4) active ratio
        total_vol = sum(r["volume"] for r in per_tet)
        active_vol = sum(r["volume"] for r in per_tet if active_mask[r["simplex_index"]])
        active_ratio = active_vol / total_vol if total_vol > 0 else 0.0

        # 5) LB_global & ms_b
        ub_active = [r for r in per_tet
                     if (ub_idx in r["vert_idx"]) and active_mask.get(r["simplex_index"], False)]
        if ub_active:
            ms_b_rec   = min(ub_active, key=lambda r: r["ms"])
            ms_b       = float(ms_b_rec["ms"])
            ms_b_simp  = int(ms_b_rec["simplex_index"])
            LB_global  = UB_global + ms_b
        else:
            ms_b       = float('nan')
            ms_b_simp  = None
            active_LBs = [r["LB"] for r in per_tet if active_mask.get(r["simplex_index"], False)]
            LB_global  = float(min(active_LBs)) if active_LBs else float(min(r["LB"] for r in per_tet))

        # 6) ms_a（active 内最小 ms）
        if any(active_mask.values()):
            ms_a = float(min(r["ms"] for r in per_tet if active_mask[r["simplex_index"]]))
        else:
            ms_a = float(min(r["ms"] for r in per_tet))
        ms_iter = ms_a

        # 7) 记录
        LB_hist.append(LB_global)
        UB_hist.append(UB_global)
        ms_hist.append(ms_iter)
        node_count.append(len(nodes))
        UB_node_hist.append(UB_node)
        ms_a_hist.append(ms_a)
        ms_b_hist.append(ms_b)
        active_ratio_hist.append(active_ratio)

        # 8) 打印
        simp_with_min = [r["simplex_index"] for r in per_tet if ub_idx in r["vert_idx"]]
        if verbose:
            print(f"[Iter {it}] Active simplex ratio = {active_ratio:.6f}")
            print(f"[Iter {it}] UB node {UB_node} is in simplices {sorted(simp_with_min)}")
            msb_src = f"T{ms_b_simp}" if ms_b_simp is not None else "N/A"
            print(f"[Iter {it}] LB = {LB_global:.6f} = UB({UB_global:.6f}) + ms_b({ms_b:.3e}) from {msb_src}")

        # 9) 候选排行
        active = [r for r in per_tet if active_mask[r["simplex_index"]]]
        ub_active = [r for r in active if ub_idx in r["vert_idx"]]
        candidates_pool = ub_active if len(ub_active) > 0 else active

        def score(cand):
            ms = cand["ms"]
            pt = cand.get("x_ms_best_scene", None)
            d  = (float('inf') if pt is None else min_dist_to_nodes(pt, nodes))
            return (ms, -d)

        candidates_sorted = sorted(candidates_pool, key=score)
        if verbose:
            top_msg = "N/A"
            if len(candidates_sorted) > 0:
                top0 = candidates_sorted[0]
                top_msg = f"T{int(top0['simplex_index'])}, ms={float(top0['ms']):.3e}"
            msb_src = f"T{ms_b_simp}" if ms_b_simp is not None else "N/A"
            print(f"[Iter {it}] LB = {LB_global:.6f} = UB({UB_global:.6f}) + ms_b({ms_b:.3e}) from {msb_src}")
            print(f"[Iter {it}] candidate rank #1: {top_msg}")

            topN = candidates_sorted[:10]
            print("== ms candidates (sorted by (ms, -dist)) ==")
            print(f"{'rank':>4} {'simp':>6} {'ms':>12} {'mind(all)':>12} {'pt':>30}")
            print("-" * 90)
            for rnk, cand in enumerate(topN, start=1):
                pt = cand.get("x_ms_best_scene", None)
                d  = (float('nan') if pt is None else min_dist_to_nodes(pt, nodes))
                pt_str = "None" if pt is None else f"({pt[0]:.4f}, {pt[1]:.4f}, {pt[2]:.4f})"
                print(f"{rnk:>4} T{cand['simplex_index']:<4} {cand['ms']:>12.4e} {d:>12.2e} {pt_str:>30}")
            print()

        # 10) 选新点 + 强校验/撞车处理
        new_node = None
        chosen_ms = None
        chosen_cand = None
        stop_due_to_collision = False

        def handle_collision(cand_pt, cand, stage_note="active"):
            nonlocal stop_due_to_collision
            X = np.asarray(nodes, float)
            P = np.asarray(cand_pt, float)
            dists = np.linalg.norm(X - P, axis=1)
            j_star = int(np.argmin(dists))
            d_star = float(dists[j_star])
            orange_ids = [r["simplex_index"] for r in per_tet if j_star in r["vert_idx"]]
            debug_pack = {
                "reason": "candidate_too_close",
                "iter": it,
                "stage": stage_note,
                "min_dist": float(min_dist),
                "closest_node_index": j_star,
                "closest_node_point": tuple(map(float, nodes[j_star])),
                "closest_distance": d_star,
                "cand_simplex": int(cand["simplex_index"]),
                "cand_point": tuple(map(float, cand_pt)),
                "cand_ms": float(cand["ms"]),
                "UB_global": float(UB_global),
                "LB_global": float(LB_global),
                "active_ratio": float(active_ratio),
                "UB_node": tuple(map(float, UB_node)),
                "active_mask": {int(k): bool(v) for k, v in active_mask.items()},
                "nodes_snapshot": [tuple(map(float, nd)) for nd in nodes],
                "per_tet_snapshot": [
                    {
                        "simplex_index": int(r["simplex_index"]),
                        "vert_idx": list(map(int, r["vert_idx"])),
                        "verts": [tuple(map(float, x)) for x in r['verts']],
                        "ms": float(r["ms"]),
                        "LB": float(r["LB"]),
                        "UB": float(r["UB"]),
                        "best_scene": int(r["best_scene"]),
                        "x_ms_best_scene": tuple(map(float, r["x_ms_best_scene"])),
                        "volume": float(r["volume"]),
                    } for r in per_tet
                ],
                "highlight_simplices": list(map(int, orange_ids)),
            }
            global LAST_DEBUG
            LAST_DEBUG = debug_pack
            plot_iteration_plotly(
                it, nodes, tri, active_mask, UB_node, cand_pt, per_tet,
                highlight_simplices=orange_ids
            )
            if verbose:
                print(
                    f"[STOP] Candidate {tuple(map(float, cand_pt))} "
                    f"is too close to existing node #{j_star} at distance {d_star:.3e} "
                    f"(< {min_dist:g}). Highlighted simplices: {sorted(orange_ids)}"
                )
            stop_due_to_collision = True

        for rank, cand in enumerate(candidates_sorted, start=1):
            cand_pt = cand.get("x_ms_best_scene", None)
            if cand_pt is None:
                continue
            if min_dist_to_nodes(cand_pt, nodes) >= min_dist:
                new_node   = cand_pt
                chosen_ms  = cand["ms"]
                chosen_cand= cand
                if verbose:
                    print(
                        f"Chosen node {tuple(map(float, cand_pt))} "
                        f"with ms={chosen_ms:.3e} "
                        f"(simp T{cand['simplex_index']}, rank #{rank})"
                    )
                    print(f"[Iter {it}] next node comes from simplex T{int(cand['simplex_index'])}")
                break
            else:
                if verbose:
                    print(
                        f"Skip candidate {tuple(map(float, cand_pt))} "
                        f"(simp T{cand['simplex_index']}, rank #{rank}) "
                        f"because too close to existing nodes (< {min_dist:g})."
                    )
                handle_collision(cand_pt, cand, stage_note="active")
                break

        if (new_node is None) and (not stop_due_to_collision) and (len(active) > 0):
            if verbose:
                print("[fallback] All active candidates too close; try all simplices...")
            all_sorted = sorted(per_tet, key=score)
            for cand in all_sorted:
                cand_pt = cand.get("x_ms_best_scene", None)
                if cand_pt is None:
                    continue
                if min_dist_to_nodes(cand_pt, nodes) >= min_dist:
                    new_node   = cand_pt
                    chosen_ms  = cand["ms"]
                    chosen_cand= cand
                    if verbose:
                        print(
                            f"Chosen node {tuple(map(float, cand_pt))} "
                            f"with ms={chosen_ms:.3e} "
                            f"(simp T{cand['simplex_index']}) [fallback-all]"
                        )
                        print(f"[Iter {it}] next node comes from simplex T{int(cand['simplex_index'])} [fallback-all]")
                    break
                else:
                    if verbose:
                        print(
                            f"Skip (all) candidate {tuple(map(float, cand_pt))} "
                            f"(simp T{cand['simplex_index']}) "
                            f"because too close to existing nodes (< {min_dist:g})."
                        )
                    handle_collision(cand_pt, cand, stage_note="fallback-all")
                    break

        if stop_due_to_collision:
            if verbose:
                print(f"[Iter {it}] Stop due to collision.")
            break

        if new_node is None:
            if verbose:
                print("New node too close for all candidates (or infeasible ms); stop.")
            break

        # == 强校验 ==
        tol_same = 1e-10
        def _same(a, b, tol=tol_same):
            a = np.asarray(a, float); b = np.asarray(b, float)
            return np.linalg.norm(a - b) <= tol

        if chosen_cand is not None:
            offending_vert = None
            for v in chosen_cand["verts"]:
                if _same(new_node, v):
                    offending_vert = tuple(map(float, v))
                    break
            if offending_vert is not None:
                LAST_DEBUG = {
                    "reason": "next_node_equals_vertex",
                    "iter": it,
                    "new_node": tuple(map(float, new_node)),
                    "offending_vertex": offending_vert,
                    "tol_same": tol_same,
                    "candidate": {
                        "simplex_index": int(chosen_cand["simplex_index"]),
                        "vert_idx": list(map(int, chosen_cand["vert_idx"])),

                        "verts": [tuple(map(float, x)) for x in chosen_cand["verts"]],
                        "ms": float(chosen_cand["ms"]),
                        "ms_per_scene": [float(x) for x in chosen_cand["ms_per_scene"]],
                        "best_scene": int(chosen_cand["best_scene"]),
                        "x_ms_best_scene": tuple(map(float, chosen_cand["x_ms_best_scene"])),
                        "LB": float(chosen_cand["LB"]),
                        "UB": float(chosen_cand["UB"]),
                        "volume": float(chosen_cand["volume"]),
                    },
                    "UB_global": float(UB_global),
                    "LB_global": float(LB_global),
                    "active_ratio": float(active_ratio),
                    "UB_node": tuple(map(float, UB_node)),
                    "active_mask": {int(k): bool(v) for k, v in active_mask.items()},
                    "nodes_snapshot": [tuple(map(float, nd)) for nd in nodes],
                    "per_tet_snapshot": [
                        {
                            "simplex_index": int(r["simplex_index"]),
                            "vert_idx": list(map(int, r["vert_idx"])),
                            "verts": [tuple(map(float, x)) for x in r["verts"]],
                            "ms": float(r["ms"]),
                            "LB": float(r["LB"]),
                            "UB": float(r["UB"]),
                            "best_scene": int(r["best_scene"]),
                            "x_ms_best_scene": tuple(map(float, r["x_ms_best_scene"])),
                            "volume": float(r["volume"]),
                        } for r in per_tet
                    ],
                }
                vert_idx_list = []
                for j, nd in enumerate(nodes):
                    if _same(offending_vert, nd):
                        vert_idx_list.append(j)
                orange_ids = [r["simplex_index"] for r in per_tet if any(j in r["vert_idx"] for j in vert_idx_list)]
                plot_iteration_plotly(it, nodes, tri, active_mask, UB_node, new_node, per_tet,
                                      highlight_simplices=orange_ids)
                if verbose:
                    print("[STOP] new_node coincides with a simplex vertex. Highlighted simplices:",
                          sorted(orange_ids))
                break

        # 可视化（正常迭代）
        plot_iteration_plotly(it, nodes, tri, active_mask, UB_node, new_node, per_tet,
                              highlight_simplices=None)

        # 加点并评估（持久化 base）
        new_vals = []
        for ω in range(S):
            val = evaluate_Q_at(base_bundles[ω], first_vars_list[ω], new_node)
            new_vals.append(val)

        nodes.append(tuple(map(float, new_node)))
        for ω in range(S):
            scen_values[ω].append(new_vals[ω])

        add_node_hist.append(new_node)
        it += 1

    return {
        "nodes": np.array(nodes, float),
        "LB_hist": LB_hist,
        "UB_hist": UB_hist,
        "ms_hist": ms_hist,
        "ms_a_hist": ms_a_hist,
        "ms_b_hist": ms_b_hist,
        "node_count": node_count,
        "UB_node_hist": UB_node_hist,
        "added_nodes": add_node_hist,
        "active_ratio_hist": active_ratio_hist,
    }

# ===================== MAIN =====================
RUN_QUICK_TEST = True  # True: 先用小规模验证

if RUN_QUICK_TEST:
    csv_path       = "data.csv"
    max_scenarios  = 2
    target_nodes   = 13
else:
    csv_path       = "data.csv"
    max_scenarios  = 99
    target_nodes   = 30

bounds = {
    "x":  (None, None),
    "u":  (None, None),
    "e":  (None, None),
    "I":  (-10, 10),
    "Kp": (0, 1),
    "Ki": (0, 1),
    "Kd": (0, 1),
}
weights = (1.0, 0.01)

# build models
model_list, first_stg_vars_list, m_tmpl_list, T = build_models_from_csv(
    csv_path, h=0.2, weights=weights, bounds=bounds,
    sp0=0.0, sp1=0.5, ku_col="tau_us", tau_col="tau_xs",
    disturb_prefix="disturbance_", setpoint_change_col="setpoint_change",
    max_scenarios=max_scenarios, skip=0
)

# gurobi 参数
gurobi_options = {
    'MIPGap': 1e-1,
    'NumericFocus': 1,
    'Presolve': 2,
    'NonConvex': 2,   # 必须
    'TimeLimit': 10,  # 可按需打开/删除
}

# 持久化封装
base_bundles, ms_bundles = build_persistent_bundles(model_list, first_stg_vars_list, gurobi_options)
# 共享λ的聚合 ms bundle
agg_bundle = AggregateMSBundle(model_list, first_stg_vars_list, gurobi_options)


# run
hist = run_pid_simplex_3d(
    base_bundles=base_bundles,
    ms_bundles=ms_bundles,            # 保留，不影响；仅当 agg_bundle=None 时才会用到
    model_list=model_list,
    first_vars_list=first_stg_vars_list,
    target_nodes=target_nodes,
    min_dist=MIN_DIST,
    active_tol=ACTIVE_TOL,
    verbose=True,
    agg_bundle=agg_bundle             # ✅ 启用共享λ
)


print("\n==== Done ====")
print(f"Total nodes: {len(hist['nodes'])}")
print(f"Best UB: {min(hist['UB_hist']) if hist['UB_hist'] else None}")
print(f"Last LB: {hist['LB_hist'][-1] if hist['LB_hist'] else None}")
# ================================================================================================================


Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2689754
Academic license 2689754 - for non-commercial use only - registered to yi___@math.ubc.ca
Set parameter MIPGap to value 0.1
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter TimeLimit to value 10
Set parameter MIPGap to value 0.1
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter TimeLimit to value 10
Set parameter MIPGap to value 0.1
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter TimeLimit to value 10
Set parameter MIPGap to value 0.1
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter TimeLimit to value 10
Set parameter MIPGap to value 0.1
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set pa

[Iter 1] Active simplex ratio = 1.000000
[Iter 1] UB node (1.0, 1.0, 1.0) is in simplices [4, 5, 8, 9, 10, 11]
[Iter 1] LB = -1.021725 = UB(0.633040) + ms_b(-1.655e+00) from T10
[Iter 1] LB = -1.021725 = UB(0.633040) + ms_b(-1.655e+00) from T10
[Iter 1] candidate rank #1: T10, ms=-1.655e+00
== ms candidates (sorted by (ms, -dist)) ==
rank   simp           ms    mind(all)                             pt
------------------------------------------------------------------------------------------
   1 T10    -1.6548e+00     4.24e-01       (0.3001, 0.3001, 1.0000)
   2 T11    -1.6548e+00     4.24e-01       (0.3001, 0.3001, 1.0000)
   3 T4     -7.7654e-01     4.17e-01       (0.2950, 1.0000, 0.2950)
   4 T5     -7.7654e-01     4.17e-01       (0.2950, 1.0000, 0.2950)

Chosen node (0.3001326151297914, 0.3001326131688806, 0.9999999990386219) with ms=-1.655e+00 (simp T10, rank #1)
[Iter 1] next node comes from simplex T10


[Iter 2] Active simplex ratio = 0.957070
[Iter 2] UB node (1.0, 1.0, 1.0) is in simplices [4, 5, 8, 9, 10, 11, 16, 17]
[Iter 2] LB = -0.143502 = UB(0.633040) + ms_b(-7.765e-01) from T8
[Iter 2] LB = -0.143502 = UB(0.633040) + ms_b(-7.765e-01) from T8
[Iter 2] candidate rank #1: T8, ms=-7.765e-01
== ms candidates (sorted by (ms, -dist)) ==
rank   simp           ms    mind(all)                             pt
------------------------------------------------------------------------------------------
   1 T8     -7.7654e-01     4.17e-01       (0.2950, 1.0000, 0.2950)
   2 T9     -7.7654e-01     4.17e-01       (0.2950, 1.0000, 0.2950)
   3 T5     -2.3482e-01     3.65e-01       (0.5582, 0.5582, 1.0000)
   4 T4     -2.3482e-01     3.65e-01       (0.5582, 0.5582, 1.0000)

Chosen node (0.29495657115827917, 0.9999999978195133, 0.29495656650945135) with ms=-7.765e-01 (simp T8, rank #1)
[Iter 2] next node comes from simplex T8


In [1]:
# ===================== ALL-IN-ONE: 3D simplex (tetrahedra) for PID-SP (UB-neighborhood per-scene ms) =====================
# Requirements: pyomo, gurobi, numpy, scipy, plotly, tqdm, csv file "data.csv"
# -----------------------------------------------------------------------------------------

import numpy as np
import itertools as it
import csv
from tqdm import tqdm
import pyomo.environ as pyo
from pyomo.opt import SolverStatus, TerminationCondition
from pyomo.solvers.plugins.solvers.gurobi_persistent import GurobiPersistent
from scipy.spatial import Delaunay
import plotly.graph_objects as go
import matplotlib.pyplot as plt

# ===== debug bucket =====
LAST_DEBUG = None   # 如果 next_node == 顶点 或 撞车，会把当时的上下文塞进来

# ------------------------- Config knobs -------------------------
MIN_DIST   = 1e-8     # 去重阈值
ACTIVE_TOL = 1e-8     # active 判定容差（此版本仅用于打印/比率）
MS_AGG     = "sum"    # 仅用于打印的标注；本版本选点不再用聚合 ms，而是(单形,场景)最小 ms

# ------------------------- PID scenario model -------------------------
def build_pid_model(T=10, h=0.2, scen=None, weights=(1.0, 0.01),
                    bounds=None, use_cvar=False, alpha=0.95):
    assert scen is not None, "请提供一个场景字典"
    Ku, tau, d, sp = scen["Ku"], scen["tau"], scen["d"], scen["sp"]
    assert len(d) == T+1 and len(sp) == T+1

    if bounds is None:
        bounds = {}
    bx = bounds.get("x",  (-20, 20))
    bu = bounds.get("u",  (None, None))
    bKp= bounds.get("Kp", (0, 10))
    bKi= bounds.get("Ki", (0, 10))
    bKd= bounds.get("Kd", (0, 10))
    be = bounds.get("e",  (-100, 100))
    bI = bounds.get("I",  (-200, 200))

    m = pyo.ConcreteModel()
    m.T  = pyo.RangeSet(0, T)
    m.Tm = pyo.RangeSet(1, T)

    m.Kp = pyo.Var(bounds=bKp)
    m.Ki = pyo.Var(bounds=bKi)
    m.Kd = pyo.Var(bounds=bKd)

    m.x = pyo.Var(m.T, bounds=bx)
    m.u = pyo.Var(m.T, bounds=bu)
    m.e = pyo.Var(m.T, bounds=be)
    m.I = pyo.Var(m.T, bounds=bI)

    # error
    def _err_rule(m, t): return m.e[t] == sp[t] - m.x[t]
    m.err_def = pyo.Constraint(m.T, rule=_err_rule)

    # integral
    def _I_dyn(m, t): return m.I[t] == m.I[t-1] + h*m.e[t]
    m.I_dyn = pyo.Constraint(m.Tm, rule=_I_dyn)

    # plant
    def _x_dyn(m, t):
        return m.x[t] == m.x[t-1] + (h/tau)*(-m.x[t] + Ku*m.u[t] + d[t])
    m.x_dyn = pyo.Constraint(m.Tm, rule=_x_dyn)

    # pid
    def _pid_rule(m, t):
        if t == 0:
            return m.u[t] == m.Kp*m.e[t] + m.Ki*m.I[t]
        return m.u[t] == m.Kp*m.e[t] + m.Ki*m.I[t] + m.Kd*(m.e[t]-m.e[t-1])/h
    m.pid = pyo.Constraint(m.T, rule=_pid_rule)

    m.x0 = pyo.Constraint(expr=m.x[0] == 0)
    m.I0 = pyo.Constraint(expr=m.I[0] == 0)

    w_e, w_u = weights
    m.cost = pyo.Expression(expr=sum(h*(w_e*m.e[t]**2 + w_u*m.u[t]**2) for t in m.T))
    m.obj_expr = pyo.Expression(expr=m.cost)  # 只保留表达式，目标在持久化阶段统一创建
    return m, [m.Kp, m.Ki, m.Kd]

def load_scenarios_from_csv(csv_path: str, T: int | None = None,
                            sp0: float = 0.0, sp1: float = 0.5,
                            ku_col: str = "tau_us", tau_col: str = "tau_xs",
                            disturb_prefix: str = "disturbance_",
                            setpoint_change_col: str = "setpoint_change"):
    scens = []
    # 推断 T
    if T is None:
        with open(csv_path, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            fields  = reader.fieldnames or []
            max_idx = -1
            for name in fields:
                if name.startswith(disturb_prefix):
                    try:
                        k = int(name[len(disturb_prefix):])
                        max_idx = max(max_idx, k)
                    except:
                        pass
            if max_idx < 0:
                raise ValueError(f"未找到扰动列前缀 {disturb_prefix}k")
            T = max_idx

    with open(csv_path, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            Ku  = float(row[ku_col])
            tau = float(row[tau_col])

            d = []
            for t in range(T+1):
                col = f"{disturb_prefix}{t}"
                d.append(float(row[col]))

            sp = [sp1]*(T+1)
            if setpoint_change_col in row and row[setpoint_change_col] != "":
                try:
                    t_star = int(float(row[setpoint_change_col]))
                    for t in range(T+1):
                        sp[t] = sp0 if t < t_star else sp1
                except:
                    pass

            scens.append({"Ku": Ku, "tau": tau, "d": d, "sp": sp})
    return scens, T

def build_models_from_csv(csv_path: str, h: float = 0.2,
                          weights=(1.0, 0.01), bounds=None,
                          sp0: float = 0.0, sp1: float = 0.5,
                          ku_col: str = "tau_us", tau_col: str = "tau_xs",
                          disturb_prefix: str = "disturbance_",
                          setpoint_change_col: str = "setpoint_change",
                          max_scenarios=None, skip=0):
    scens, T = load_scenarios_from_csv(
        csv_path=csv_path, T=None, sp0=sp0, sp1=sp1,
        ku_col=ku_col, tau_col=tau_col,
        disturb_prefix=disturb_prefix,
        setpoint_change_col=setpoint_change_col,
    )
    if skip or max_scenarios:
        scens = scens[skip: (skip + max_scenarios) if max_scenarios else None]

    model_list, first_stg_vars_list = [], []
    for scen in scens:
        m, yvars = build_pid_model(T=T, h=h, scen=scen, weights=weights, bounds=bounds)
        model_list.append(m)
        first_stg_vars_list.append(yvars)

    m_tmpl_list = [model_list[0], first_stg_vars_list[0]]
    return model_list, first_stg_vars_list, m_tmpl_list, T

# ------------------------- Persistent wrappers -------------------------
class BaseBundle:
    """每个场景的基础模型（计算真实Q）+ 持久化求解器"""
    def __init__(self, model: pyo.ConcreteModel, options: dict | None = None):
        self.model = model
        self.gp = GurobiPersistent()
        self.gp.set_instance(model)
        if hasattr(model, 'obj'):
            model.del_component('obj')
        model.obj = pyo.Objective(expr=model.obj_expr, sense=pyo.minimize)
        self.gp.set_objective(model.obj)
        if options:
            self.gp.set_gurobi_param('MIPGap', options.get('MIPGap', 1e-1))
            self.gp.set_gurobi_param('NumericFocus', options.get('NumericFocus', 1))
            self.gp.set_gurobi_param('Presolve', options.get('Presolve', 2))
            self.gp.set_gurobi_param('NonConvex', options.get('NonConvex', 2))
            if 'TimeLimit' in options:
                self.gp.set_gurobi_param('TimeLimit', options['TimeLimit'])

    def eval_at(self, first_vars, first_vals):
        for v, val in zip(first_vars, first_vals):
            v.fix(float(val))
            self.gp.update_var(v)
        self.gp.solve(load_solutions=True)
        val = float(pyo.value(self.model.obj_expr))
        for v in first_vars:
            v.unfix()
            self.gp.update_var(v)
        return val

class MSBundle:
    """ms 模板（单场景，持久化）：每次更新四面体时 remove+add 约束，兼容无 set_* API 的 Pyomo."""
    def __init__(self, model_base: pyo.ConcreteModel, first_vars, options: dict | None = None):
        # 克隆该场景模型
        m = model_base.clone()

        # 重心变量与和=1
        m.lam_index = pyo.RangeSet(0, 3)
        m.lam = pyo.Var(m.lam_index, domain=pyo.NonNegativeReals)
        m.lam_sum = pyo.Constraint(expr=sum(m.lam[j] for j in m.lam_index) == 1.0)

        # 一阶段变量引用
        self.Kp = m.find_component(first_vars[0].name)
        self.Ki = m.find_component(first_vars[1].name)
        self.Kd = m.find_component(first_vars[2].name)
        if any(v is None for v in (self.Kp, self.Ki, self.Kd)):
            raise RuntimeError("克隆模型中找不到 Kp/Ki/Kd")

        # 先放占位约束（系数0），真正的表达式在 update_tetra 里重建
        m.link_kp = pyo.Constraint(expr=self.Kp == sum(0.0 * m.lam[j] for j in m.lam_index))
        m.link_ki = pyo.Constraint(expr=self.Ki == sum(0.0 * m.lam[j] for j in m.lam_index))
        m.link_kd = pyo.Constraint(expr=self.Kd == sum(0.0 * m.lam[j] for j in m.lam_index))

        m.As = pyo.Var()
        m.As_def = pyo.Constraint(expr=m.As == sum(0.0 * m.lam[j] for j in m.lam_index))

        # 目标
        if hasattr(m, 'obj'):
            m.del_component('obj')
        m.obj = pyo.Objective(expr=m.obj_expr - m.As, sense=pyo.minimize)

        # 持久化求解器
        self.model = m
        self.gp = GurobiPersistent()
        self.gp.set_instance(m)
        self.gp.set_objective(m.obj)
        if options:
            self.gp.set_gurobi_param('MIPGap', options.get('MIPGap', 1e-1))
            self.gp.set_gurobi_param('NumericFocus', options.get('NumericFocus', 1))
            self.gp.set_gurobi_param('Presolve', options.get('Presolve', 2))
            self.gp.set_gurobi_param('NonConvex', options.get('NonConvex', 2))
            if 'TimeLimit' in options:
                self.gp.set_gurobi_param('TimeLimit', options['TimeLimit'])

        # 缓存/别名
        self.lam = m.lam
        self.link_kp = m.link_kp
        self.link_ki = m.link_ki
        self.link_kd = m.link_kd
        self.As     = m.As
        self.As_def = m.As_def
        self._V_cached = None  # [(x,y,z)]*4

    # ---- 工具：构建表达式 & 热替换约束（先从求解器移除，再从模型删掉，重建并 add 回去） ----
    def _expr_link(self, lhs_var, coeffs):
        return lhs_var == sum(float(coeffs[j]) * self.lam[j] for j in range(4))

    def _replace_constraint(self, attr_name: str, expr):
        # 1) 从求解器里移除旧约束（若已存在）
        old_con = getattr(self.model, attr_name)
        try:
            self.gp.remove_constraint(old_con)
        except Exception:
            pass  # 第一次可能还没被单独 add 过

        # 2) 从模型里删除旧组件
        self.model.del_component(old_con)

        # 3) 以相同名字重建新约束并加入模型
        new_con = pyo.Constraint(expr=expr)
        self.model.add_component(attr_name, new_con)

        # 4) 通知持久化求解器新增该约束
        self.gp.add_constraint(getattr(self.model, attr_name))

    # ---- 外部接口：写入四面体数据并求解 ----
    def update_tetra(self, tet_vertices, fverts_scene):
        # 稳定排序（与外层一致）
        pairs = sorted(
            [(tuple(map(float, tet_vertices[j])), float(fverts_scene[j])) for j in range(4)],
            key=lambda kv: (kv[0][0], kv[0][1], kv[0][2])
        )
        V = [kv[0] for kv in pairs]  # 顶点坐标
        F = [kv[1] for kv in pairs]  # 顶点值
        self._V_cached = V

        vx = [V[j][0] for j in range(4)]
        vy = [V[j][1] for j in range(4)]
        vz = [V[j][2] for j in range(4)]

        # 用 remove+add 的方式重建四条等式
        self._replace_constraint('link_kp', self._expr_link(self.Kp, vx))
        self._replace_constraint('link_ki', self._expr_link(self.Ki, vy))
        self._replace_constraint('link_kd', self._expr_link(self.Kd, vz))
        self._replace_constraint('As_def',  self._expr_link(self.As, F))

    def solve(self):
        res = self.gp.solve(load_solutions=True)
        ok = (res.solver.status == SolverStatus.ok) and \
             (res.solver.termination_condition in {
                 TerminationCondition.optimal,
                 TerminationCondition.locallyOptimal
             })
        return ok

    def get_ms_and_point(self):
        ms_val = float(pyo.value(self.model.obj))
        lam_star = np.array([pyo.value(self.lam[j]) for j in range(4)], dtype=float)
        V = np.array(self._V_cached, dtype=float)
        new_pt = lam_star @ V
        return ms_val, lam_star, tuple(map(float, new_pt))

def build_persistent_bundles(model_list, first_stg_vars_list, gurobi_options: dict):
    base_bundles = []
    ms_bundles   = []
    for m, yvars in zip(model_list, first_stg_vars_list):
        base_bundles.append(BaseBundle(m, gurobi_options))
        ms_bundles.append(MSBundle(m, yvars, gurobi_options))
    return base_bundles, ms_bundles

# ------------------------- Basic utils -------------------------
def corners_from_var_bounds(vars_3):
    bnds = []
    for v in vars_3:
        lb, ub = v.lb, v.ub
        if lb is None or ub is None:
            raise ValueError(f"{v.name} 缺少上下界")
        bnds.append((float(lb), float(ub)))
    return [tuple(p) for p in it.product(*[(lo, hi) for (lo,hi) in bnds])]

def too_close(p, nodes, tol=MIN_DIST):
    return any(np.linalg.norm(np.asarray(p)-np.asarray(q)) < tol for q in nodes)

def evaluate_Q_at(base_bundle: BaseBundle, first_stg_vars, first_stg_vals):
    return base_bundle.eval_at(first_stg_vars, first_stg_vals)

def tet_volume(verts):
    V = np.array(verts, float)
    v0, v1, v2, v3 = V
    return float(abs(np.linalg.det(np.stack([v1 - v0, v2 - v0, v3 - v0], axis=1))) / 6.0)

def tet_quality(verts):
    V = np.array(verts, float)
    edges = [np.linalg.norm(V[i] - V[j]) for (i, j) in it.combinations(range(4), 2)]
    denom = float(np.sum(np.power(edges, 3))) + 1e-16
    vol = tet_volume(verts)
    return float(6.0 * vol / denom)

# ------------------------- Single tetra & scene: ms solve (persistent) -------------------------
def ms_on_tetra_for_scene(ms_bundle: MSBundle, tet_vertices, fverts_scene):
    ms_bundle.update_tetra(tet_vertices, fverts_scene)
    ok = ms_bundle.solve()
    if not ok:
        return float('inf'), None, None
    ms_val, lam_star, new_pt = ms_bundle.get_ms_and_point()
    return ms_val, lam_star, new_pt

# ------------------------- UB-neighborhood per-scene ms (new) -------------------------
def best_candidate_from_ub_neighborhood(nodes, scen_values, ms_bundles, ub_idx):
    """
    只在“包含 UB 节点”的单形里、对“每个场景”分别求 ms，
    返回 (best_cand, tri, per_tet_subset)：
      - best_cand: dict(ms, pt, simplex_index, scene, verts, vert_idx)
      - tri: Delaunay 结构（供画图）
      - per_tet_subset: 只含 UB 邻域单形的简要信息，用于可视化/打印
    """
    pts = np.asarray(nodes, float)
    if len(pts) < 4:
        return None, None, []

    tri = Delaunay(pts)
    S = len(ms_bundles)

    # 找出所有“包含 UB 节点”的单形 id 列表
    ub_sids = []
    for k, simp in enumerate(tri.simplices):
        idxs = list(map(int, simp))
        if ub_idx in idxs:
            ub_sids.append((k, idxs))

    if not ub_sids:
        return None, tri, []

    best = None
    per_tet_subset = []

    for k, idxs in ub_sids:
        verts = [tuple(pts[i]) for i in idxs]
        fverts_per_scene = [[scen_values[w][i] for i in idxs] for w in range(S)]

        for w in range(S):
            ms_val, lam_star, new_pt = ms_on_tetra_for_scene(
                ms_bundles[w], verts, fverts_per_scene[w]
            )
            if (new_pt is None) or (not np.isfinite(ms_val)):
                continue

            cand = {
                "simplex_index": k,
                "vert_idx": idxs,
                "verts": verts,
                "scene": w,
                "ms": float(ms_val),
                "pt": tuple(map(float, new_pt)),
            }
            # 维护当前全局最优（按 ms 升序；若并列，取更远离现有点）
            def mind(pt): return min_dist_to_nodes(pt, nodes)
            if (best is None) or (cand["ms"] < best["ms"] - 1e-15) \
               or (abs(cand["ms"] - best["ms"]) <= 1e-15 and mind(cand["pt"]) > mind(best["pt"])):
                best = cand

        per_tet_subset.append({
            "simplex_index": k,
            "vert_idx": idxs,
            "verts": verts,
            "LB": float("nan"),
            "UB": float("nan"),
            "ms": float("nan"),
            "volume": tet_volume(verts),
            "x_ms_best_scene": None,
            "best_scene": -1,
        })

    return best, tri, per_tet_subset

# ------------------------- Pretty print -------------------------
def print_tetra_table(per_tet, active_mask, purple_set=None, prec=6):
    purple_set = set() if purple_set is None else set(purple_set)
    per_tet = sorted(per_tet, key=lambda r: r["simplex_index"])
    tet_ids = [r["simplex_index"] for r in per_tet]
    active_set = {tid for tid in tet_ids if active_mask.get(tid, False)}

    def _mark(tid):
        s = f"T{tid}"
        flags = []
        if tid in active_set:  flags.append("*")
        if tid in purple_set:  flags.append("^")
        return s + ("".join(flags) if flags else "")

    header = ["row\\simp"] + [_mark(tid) for tid in tet_ids]
    rows = [
        ["UB"] + [f"{r['UB']:.{prec}f}" if np.isfinite(r["UB"]) else "nan" for r in per_tet],
        ["LB"] + [f"{r['LB']:.{prec}f}" if np.isfinite(r["LB"]) else "nan" for r in per_tet],
        ["ms"] + [f"{r['ms']:.3e}"       if np.isfinite(r["ms"]) else "nan" for r in per_tet],
    ]
    table = [header] + rows
    colw = [max(len(str(row[c])) for row in table) + 2 for c in range(len(header))]

    RED, PURPLE, RESET = "\033[31m", "\033[35m", "\033[0m"
    def colorize(col_idx, s):
        if col_idx == 0:
            return s
        tid = tet_ids[col_idx-1]
        if tid in purple_set:
            return f"{PURPLE}{s}{RESET}"
        elif tid in active_set:
            return f"{RED}{s}{RESET}"
        return s

    print("\n== Per-tetra summary ==")
    print("".join(colorize(c, str(header[c]).ljust(colw[c])) for c in range(len(header))))
    print("-"*sum(colw))
    for r in rows:
        line = []
        for c in range(len(header)):
            cell = str(r[c])
            pad  = cell.ljust(colw[c]) if c==0 else cell.rjust(colw[c])
            line.append(colorize(c, pad))
        print("".join(line))
    print("(红色列=active；紫色列=包含当前最小节点的单形；第1行=UB，第2行=LB，第3行=ms)\n")

def min_dist_to_nodes(pt, nodes):
    P = np.asarray(pt, float)
    X = np.asarray(nodes, float)
    return float(np.min(np.linalg.norm(X - P, axis=1)))

def print_per_scenario_ms(per_tet, max_scenarios_to_print=10, prec=3):
    per_tet = sorted(per_tet, key=lambda r: r["simplex_index"])
    if not per_tet or "ms_per_scene" not in per_tet[0]:
        return
    S = len(per_tet[0]["ms_per_scene"])
    show = min(S, max_scenarios_to_print)
    head = "simp | " + " ".join([f"s{j}".rjust(10) for j in range(show)])
    print("== Per-tetra per-scenario ms (showing first", show, "of", S, "scenes) ==")
    print(head); print("-"*len(head))
    for r in per_tet:
        arr = r["ms_per_scene"][:show]
        sline = " ".join([f"{v:.{prec}e}".rjust(10) for v in arr])
        print(f"{r['simplex_index']:>4d} | {sline}")
    if show < S:
        print(f"... ({S-show} scenes omitted)")
    print()

# ------------------------- Plotly visualization -------------------------
def plot_iteration_plotly(iter_id, nodes, tri, active_mask, ub_node, next_node, per_tet,
                          highlight_simplices=None):
    import numpy as np
    import plotly.graph_objects as go

    if highlight_simplices is None:
        highlight_simplices = set()
    else:
        highlight_simplices = set(highlight_simplices)

    fig = go.Figure()
    nodes = np.asarray(nodes, float)

    if len(nodes) > 0:
        fig.add_trace(go.Scatter3d(
            x=nodes[:, 0], y=nodes[:, 1], z=nodes[:, 2],
            mode='markers',
            marker=dict(size=4, color="black"),
            name='nodes'
        ))

    if ub_node is not None:
        fig.add_trace(go.Scatter3d(
            x=[ub_node[0]], y=[ub_node[1]], z=[ub_node[2]],
            mode='markers',
            marker=dict(size=7, symbol="circle", color="green"),
            name='current min node'
        ))

    if next_node is not None:
        fig.add_trace(go.Scatter3d(
            x=[next_node[0]], y=[next_node[1]], z=[next_node[2]],
            mode='markers',
            marker=dict(size=8, symbol="diamond", color="#1976d2"),
            name='next node'
        ))

    def _is_same_point(a, b, atol=1e-6):
        if a is None or b is None:
            return False
        return np.linalg.norm(np.asarray(a, float) - np.asarray(b, float)) <= float(atol)

    if tri is not None:
        legend_mesh_added = False
        legend_edge_added = False

        for r in per_tet:
            sid = r["simplex_index"]
            if not active_mask.get(sid, False):
                continue

            verts = np.array(r["verts"], dtype=float)
            highlight_by_next = _is_same_point(next_node, r.get("x_ms_best_scene", None), atol=1e-6)
            highlight = highlight_by_next or (sid in highlight_simplices)

            mesh_color = "#ff5722" if highlight else "#ffb74d"
            edge_color = "darkorange"
            edge_width = 4 if highlight else 3
            mesh_opacity = 0.45 if highlight else 0.35

            I = [0, 0, 0, 1]
            J = [1, 1, 2, 2]
            K = [2, 3, 3, 3]

            fig.add_trace(go.Mesh3d(
                x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
                i=I, j=J, k=K,
                color=mesh_color,
                opacity=mesh_opacity,
                showscale=False,
                name="active simplex",
                showlegend=(not legend_mesh_added)
            ))
            legend_mesh_added = True

            edges = [(0,1), (0,2), (0,3), (1,2), (1,3), (2,3)]
            for (a, b) in edges:
                pa, pb = verts[a], verts[b]
                fig.add_trace(go.Scatter3d(
                    x=[pa[0], pb[0]],
                    y=[pa[1], pb[1]],
                    z=[pa[2], pb[2]],
                    mode='lines',
                    line=dict(width=edge_width, color=edge_color),
                    name='active edge',
                    showlegend=(not legend_edge_added)
                ))
            legend_edge_added = True

            cx, cy, cz = np.mean(verts, axis=0)
            qtxt = ""
            if "quality" in r and r["quality"] is not None:
                try:
                    qtxt = f"<br>q={float(r['quality']):.3e}"
                except Exception:
                    qtxt = ""
            txt = (f"simp={sid}"
                   f"<br>vol={float(r['volume']):.3e}"
                   f"{qtxt}")

            fig.add_trace(go.Scatter3d(
                x=[cx], y=[cy], z=[cz],
                mode='markers',
                marker=dict(size=1, opacity=0.0),
                text=[txt], hoverinfo="text",
                name="tetra info",
                showlegend=False
            ))

    fig.update_layout(
        title=f"Iteration {iter_id}",
        scene=dict(
            xaxis_title="Kp",
            yaxis_title="Ki",
            zaxis_title="Kd",
            aspectmode="cube",
            zaxis=dict(tickformat=".2f"),
        ),
        width=980,
        height=720,
        legend=dict(itemsizing="constant")
    )
    fig.update_traces(
        hovertemplate="x: %{x:.6f}<br>y: %{y:.6f}<br>z: %{z:.6f}",
        selector=dict(type='scatter3d')
    )
    fig.show()

# ------------------------- MAIN LOOP (modified selection strategy) -------------------------
def run_pid_simplex_3d(base_bundles, ms_bundles, model_list, first_vars_list,
                       target_nodes=30, min_dist=MIN_DIST, active_tol=ACTIVE_TOL, verbose=True):
    global LAST_DEBUG
    LB_hist, UB_hist, ms_hist, node_count = [], [], [], []
    UB_node_hist, add_node_hist = [], []
    ms_a_hist, ms_b_hist = [], []
    active_ratio_hist = []

    S = len(model_list)
    nodes = corners_from_var_bounds(first_vars_list[0])

    bounds_arr = np.array([[float(v.lb), float(v.ub)] for v in first_vars_list[0]], float)
    diam = float(np.linalg.norm(bounds_arr[:,1] - bounds_arr[:,0]))
    min_dist = float(min_dist)

    # 缓存 f_ω(node_i)
    scen_values = [[None]*len(nodes) for _ in range(S)]
    for i, node in enumerate(nodes):
        for ω in range(S):
            scen_values[ω][i] = evaluate_Q_at(base_bundles[ω], first_vars_list[ω], node)

    it = 0
    while len(nodes) < target_nodes:
        # 1) 全局 UB
        f_sum_per_node = [
            sum(scen_values[ω][i] for ω in range(S))
            for i in range(len(nodes))
        ]
        ub_idx = int(np.argmin(f_sum_per_node))
        UB_global = float(f_sum_per_node[ub_idx])
        UB_node = tuple(nodes[ub_idx])

        # 2) 只在 UB 邻域求解逐场景 ms，拿全局最小的那个候选
        best_cand, tri, ub_per_tet = best_candidate_from_ub_neighborhood(
            nodes, scen_values, ms_bundles, ub_idx
        )
        if best_cand is None:
            if verbose:
                print("No candidate found in UB neighborhood; stop.")
            break

        # 近似的 LB：用 UB + (该邻域里的最小 ms) —— 与原义 LB=UB+ms_b 保持一致
        LB_global = UB_global + best_cand["ms"]

        # 打印（active ratio 用 UB 邻域体积 / 总体积）
        total_vol = sum(tet_volume([tuple(nodes[i]) for i in simp]) for simp in tri.simplices)
        active_vol = sum(r["volume"] for r in ub_per_tet)
        active_ratio = (active_vol / total_vol) if total_vol > 0 else 0.0

        LB_hist.append(LB_global)
        UB_hist.append(UB_global)
        ms_hist.append(best_cand["ms"])
        node_count.append(len(nodes))
        UB_node_hist.append(UB_node)
        ms_a_hist.append(best_cand["ms"])
        ms_b_hist.append(best_cand["ms"])
        active_ratio_hist.append(active_ratio)

        if verbose:
            ub_sids = sorted([r["simplex_index"] for r in ub_per_tet])
            print(f"[Iter {it}] Active(UB-neighborhood) simplex ratio = {active_ratio:.6f}")
            print(f"[Iter {it}] UB node {UB_node} is in simplices {ub_sids}")
            print(f"[Iter {it}] LB ≈ {LB_global:.6f} = UB({UB_global:.6f}) + min_ms({best_cand['ms']:.3e}) "
                  f"at T{best_cand['simplex_index']} (scene {best_cand['scene']})")

        # 3) 选点与防撞（用 best_cand）
        new_node = best_cand["pt"]

        # 撞车处理
        if min_dist_to_nodes(new_node, nodes) < min_dist:
            X = np.asarray(nodes, float)
            P = np.asarray(new_node, float)
            dists = np.linalg.norm(X - P, axis=1)
            j_star = int(np.argmin(dists))
            d_star = float(dists[j_star])

            orange_ids = [r["simplex_index"] for r in ub_per_tet if j_star in r["vert_idx"]]
            global LAST_DEBUG
            LAST_DEBUG = {
                "reason": "candidate_too_close",
                "iter": it,
                "stage": "ub-neighborhood",
                "min_dist": float(min_dist),
                "closest_node_index": j_star,
                "closest_node_point": tuple(map(float, nodes[j_star])),
                "closest_distance": d_star,
                "cand_simplex": int(best_cand["simplex_index"]),
                "cand_scene": int(best_cand["scene"]),
                "cand_point": tuple(map(float, new_node)),
                "cand_ms": float(best_cand["ms"]),
                "UB_global": float(UB_global),
                "LB_global": float(LB_global),
                "active_ratio": float(active_ratio),
                "UB_node": tuple(map(float, UB_node)),
                "nodes_snapshot": [tuple(map(float, nd)) for nd in nodes],
                "highlight_simplices": list(map(int, orange_ids)),
            }
            active_mask = {r["simplex_index"]: True for r in ub_per_tet}
            plot_iteration_plotly(it, nodes, tri, active_mask, UB_node, new_node, ub_per_tet,
                                  highlight_simplices=orange_ids)
            if verbose:
                print(
                    f"[STOP] Candidate {tuple(map(float, new_node))} "
                    f"is too close to existing node #{j_star} at distance {d_star:.3e} "
                    f"(< {min_dist:g}). Highlighted simplices: {sorted(orange_ids)}"
                )
            break

        # 强校验：是否与来源单形的顶点重合
        tol_same = 1e-10
        def _same(a, b, tol=tol_same):
            a = np.asarray(a, float); b = np.asarray(b, float)
            return np.linalg.norm(a - b) <= tol

        offending_vert = None
        for v in best_cand["verts"]:
            if _same(new_node, v):
                offending_vert = tuple(map(float, v))
                break
        if offending_vert is not None:
            LAST_DEBUG = {
                "reason": "next_node_equals_vertex",
                "iter": it,
                "new_node": tuple(map(float, new_node)),
                "offending_vertex": offending_vert,
                "candidate": {
                    "simplex_index": int(best_cand["simplex_index"]),
                    "scene": int(best_cand["scene"]),
                    "ms": float(best_cand["ms"]),
                    "verts": [tuple(map(float, x)) for x in best_cand["verts"]],
                    "vert_idx": list(map(int, best_cand["vert_idx"])),
                },
                "UB_global": float(UB_global),
                "LB_global": float(LB_global),
                "UB_node": tuple(map(float, UB_node)),
                "nodes_snapshot": [tuple(map(float, nd)) for nd in nodes],
            }
            # 高亮与该顶点相邻的 UB 邻域单形
            vert_idx_list = []
            for j, nd in enumerate(nodes):
                if _same(offending_vert, nd):
                    vert_idx_list.append(j)
            orange_ids = [r["simplex_index"] for r in ub_per_tet if any(j in r["vert_idx"] for j in vert_idx_list)]
            active_mask = {r["simplex_index"]: True for r in ub_per_tet}
            plot_iteration_plotly(it, nodes, tri, active_mask, UB_node, new_node, ub_per_tet,
                                  highlight_simplices=orange_ids)
            if verbose:
                print("[STOP] new_node coincides with a simplex vertex. Highlighted simplices:",
                      sorted(orange_ids))
            break

        # 可视化（UB 邻域当作 active）
        active_mask = {r["simplex_index"]: True for r in ub_per_tet}
        plot_iteration_plotly(it, nodes, tri, active_mask, UB_node, new_node, ub_per_tet,
                              highlight_simplices=None)

        # 加点并评估（持久化 base）
        new_vals = []
        for ω in range(S):
            val = evaluate_Q_at(base_bundles[ω], first_vars_list[ω], new_node)
            new_vals.append(val)

        nodes.append(tuple(map(float, new_node)))
        for ω in range(S):
            scen_values[ω].append(new_vals[ω])

        add_node_hist.append(new_node)
        it += 1

    return {
        "nodes": np.array(nodes, float),
        "LB_hist": LB_hist,
        "UB_hist": UB_hist,
        "ms_hist": ms_hist,
        "ms_a_hist": ms_a_hist,
        "ms_b_hist": ms_b_hist,
        "node_count": node_count,
        "UB_node_hist": UB_node_hist,
        "added_nodes": add_node_hist,
        "active_ratio_hist": active_ratio_hist,
    }

# ===================== MAIN =====================
RUN_QUICK_TEST = True  # True: 先用小规模验证

if RUN_QUICK_TEST:
    csv_path       = "data.csv"
    max_scenarios  = 1   # 你可以调大以测试多场景
    target_nodes   = 30
else:
    csv_path       = "data.csv"
    max_scenarios  = 10
    target_nodes   = 30

bounds = {
    "x":  (None, None),
    "u":  (None, None),
    "e":  (None, None),
    "I":  (-10, 10),
    "Kp": (0, 1),
    "Ki": (0, 1),
    "Kd": (0, 1),
}
weights = (1.0, 0.01)

# build models
model_list, first_stg_vars_list, m_tmpl_list, T = build_models_from_csv(
    csv_path, h=0.2, weights=weights, bounds=bounds,
    sp0=0.0, sp1=0.5, ku_col="tau_us", tau_col="tau_xs",
    disturb_prefix="disturbance_", setpoint_change_col="setpoint_change",
    max_scenarios=max_scenarios, skip=0
)

# gurobi 参数
gurobi_options = {
    'MIPGap': 1e-1,
    'NumericFocus': 1,
    'Presolve': 2,
    'NonConvex': 2,   # 必须
    'TimeLimit': 10,  # 可按需打开/删除
}

# 持久化封装
base_bundles, ms_bundles = build_persistent_bundles(model_list, first_stg_vars_list, gurobi_options)

# run（仅在 UB 邻域 × 各场景 解 ms，选 (单形,场景) 最小）
hist = run_pid_simplex_3d(
    base_bundles=base_bundles,
    ms_bundles=ms_bundles,
    model_list=model_list,
    first_vars_list=first_stg_vars_list,
    target_nodes=target_nodes,
    min_dist=MIN_DIST,
    active_tol=ACTIVE_TOL,
    verbose=True
)

print("\n==== Done ====")
print(f"Total nodes: {len(hist['nodes'])}")
print(f"Best UB: {min(hist['UB_hist']) if hist['UB_hist'] else None}")
print(f"Last LB: {hist['LB_hist'][-1] if hist['LB_hist'] else None}")
# ================================================================================================================


Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2689754
Academic license 2689754 - for non-commercial use only - registered to yi___@math.ubc.ca
Set parameter MIPGap to value 0.1
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter TimeLimit to value 10
Set parameter MIPGap to value 0.1
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter TimeLimit to value 10
[Iter 0] Active(UB-neighborhood) simplex ratio = 0.333333
[Iter 0] UB node (1.0, 1.0, 1.0) is in simplices [2, 3]
[Iter 0] LB ≈ 0.162061 = UB(0.361626) + min_ms(-1.996e-01) at T2 (scene 0)


[Iter 1] Active(UB-neighborhood) simplex ratio = 0.420133
[Iter 1] UB node (1.0, 1.0, 1.0) is in simplices [2, 3, 8, 9, 10, 11]
[Iter 1] LB ≈ -0.558282 = UB(0.361626) + min_ms(-9.199e-01) at T2 (scene 0)


[Iter 2] Active(UB-neighborhood) simplex ratio = 0.354660
[Iter 2] UB node (1.0, 1.0, 1.0) is in simplices [4, 5, 8, 9, 10, 11, 16, 17]
[Iter 2] LB ≈ -0.558282 = UB(0.361626) + min_ms(-9.199e-01) at T16 (scene 0)


[STOP] Candidate (0.3116794713548105, 0.3116794695432848, 0.9999999992060868) is too close to existing node #9 at distance 1.459e-09 (< 1e-08). Highlighted simplices: [4, 5, 16, 17]

==== Done ====
Total nodes: 10
Best UB: 0.3616264563470583
Last LB: -0.5582819190743948


In [ ]:
#以下版本成功的使用了presistent，好像处理好了多场景问题了，现在搞ms缓存

In [4]:
# ===================== ALL-IN-ONE: 3D simplex (tetrahedra) for PID-SP with gurobi_persistent =====================
# Requirements: pyomo, gurobi, numpy, scipy, plotly, tqdm, csv file "data.csv"
# -----------------------------------------------------------------------------------------

import numpy as np
import itertools as it
import csv
from tqdm import tqdm
import pyomo.environ as pyo
from pyomo.opt import SolverStatus, TerminationCondition
from pyomo.solvers.plugins.solvers.gurobi_persistent import GurobiPersistent
from scipy.spatial import Delaunay
import plotly.graph_objects as go
import matplotlib.pyplot as plt

# ===== debug bucket =====
LAST_DEBUG = None   # 如果 next_node == 顶点 或 撞车，会把当时的上下文塞进来

# ------------------------- Config knobs -------------------------
MIN_DIST   = 1e-8     # 去重阈值
ACTIVE_TOL = 1e-8     # active 判定容差
MS_AGG     = "sum"    # 单形 ms 聚合：'sum' 或 'mean'
MS_CACHE_ENABLE = True  # <== 新增：开启/关闭 ms 结果缓存


# ------------------------- PID scenario model -------------------------
def build_pid_model(T=10, h=0.2, scen=None, weights=(1.0, 0.01),
                    bounds=None, use_cvar=False, alpha=0.95):
    assert scen is not None, "请提供一个场景字典"
    Ku, tau, d, sp = scen["Ku"], scen["tau"], scen["d"], scen["sp"]
    assert len(d) == T+1 and len(sp) == T+1

    if bounds is None:
        bounds = {}
    bx = bounds.get("x",  (-20, 20))
    bu = bounds.get("u",  (None, None))
    bKp= bounds.get("Kp", (0, 10))
    bKi= bounds.get("Ki", (0, 10))
    bKd= bounds.get("Kd", (0, 10))
    be = bounds.get("e",  (-100, 100))
    bI = bounds.get("I",  (-200, 200))

    m = pyo.ConcreteModel()
    m.T  = pyo.RangeSet(0, T)
    m.Tm = pyo.RangeSet(1, T)

    m.Kp = pyo.Var(bounds=bKp)
    m.Ki = pyo.Var(bounds=bKi)
    m.Kd = pyo.Var(bounds=bKd)

    m.x = pyo.Var(m.T, bounds=bx)
    m.u = pyo.Var(m.T, bounds=bu)
    m.e = pyo.Var(m.T, bounds=be)
    m.I = pyo.Var(m.T, bounds=bI)

    # error
    def _err_rule(m, t): return m.e[t] == sp[t] - m.x[t]
    m.err_def = pyo.Constraint(m.T, rule=_err_rule)

    # integral
    def _I_dyn(m, t): return m.I[t] == m.I[t-1] + h*m.e[t]
    m.I_dyn = pyo.Constraint(m.Tm, rule=_I_dyn)

    # plant
    def _x_dyn(m, t):
        return m.x[t] == m.x[t-1] + (h/tau)*(-m.x[t] + Ku*m.u[t] + d[t])
    m.x_dyn = pyo.Constraint(m.Tm, rule=_x_dyn)

    # pid
    def _pid_rule(m, t):
        if t == 0:
            return m.u[t] == m.Kp*m.e[t] + m.Ki*m.I[t]
        return m.u[t] == m.Kp*m.e[t] + m.Ki*m.I[t] + m.Kd*(m.e[t]-m.e[t-1])/h
    m.pid = pyo.Constraint(m.T, rule=_pid_rule)

    m.x0 = pyo.Constraint(expr=m.x[0] == 0)
    m.I0 = pyo.Constraint(expr=m.I[0] == 0)

    w_e, w_u = weights
    m.cost = pyo.Expression(expr=sum(h*(w_e*m.e[t]**2 + w_u*m.u[t]**2) for t in m.T))
    m.obj_expr = pyo.Expression(expr=m.cost)  # 只保留表达式，目标在持久化阶段统一创建
    return m, [m.Kp, m.Ki, m.Kd]

def load_scenarios_from_csv(csv_path: str, T: int | None = None,
                            sp0: float = 0.0, sp1: float = 0.5,
                            ku_col: str = "tau_us", tau_col: str = "tau_xs",
                            disturb_prefix: str = "disturbance_",
                            setpoint_change_col: str = "setpoint_change"):
    scens = []
    # 推断 T
    if T is None:
        with open(csv_path, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            fields  = reader.fieldnames or []
            max_idx = -1
            for name in fields:
                if name.startswith(disturb_prefix):
                    try:
                        k = int(name[len(disturb_prefix):])
                        max_idx = max(max_idx, k)
                    except:
                        pass
            if max_idx < 0:
                raise ValueError(f"未找到扰动列前缀 {disturb_prefix}k")
            T = max_idx

    with open(csv_path, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            Ku  = float(row[ku_col])
            tau = float(row[tau_col])

            d = []
            for t in range(T+1):
                col = f"{disturb_prefix}{t}"
                d.append(float(row[col]))

            sp = [sp1]*(T+1)
            if setpoint_change_col in row and row[setpoint_change_col] != "":
                try:
                    t_star = int(float(row[setpoint_change_col]))
                    for t in range(T+1):
                        sp[t] = sp0 if t < t_star else sp1
                except:
                    pass

            scens.append({"Ku": Ku, "tau": tau, "d": d, "sp": sp})
    return scens, T

def build_models_from_csv(csv_path: str, h: float = 0.2,
                          weights=(1.0, 0.01), bounds=None,
                          sp0: float = 0.0, sp1: float = 0.5,
                          ku_col: str = "tau_us", tau_col: str = "tau_xs",
                          disturb_prefix: str = "disturbance_",
                          setpoint_change_col: str = "setpoint_change",
                          max_scenarios=None, skip=0):
    scens, T = load_scenarios_from_csv(
        csv_path=csv_path, T=None, sp0=sp0, sp1=sp1,
        ku_col=ku_col, tau_col=tau_col,
        disturb_prefix=disturb_prefix,
        setpoint_change_col=setpoint_change_col,
    )
    if skip or max_scenarios:
        scens = scens[skip: (skip + max_scenarios) if max_scenarios else None]

    model_list, first_stg_vars_list = [], []
    for scen in scens:
        m, yvars = build_pid_model(T=T, h=h, scen=scen, weights=weights, bounds=bounds)
        model_list.append(m)
        first_stg_vars_list.append(yvars)

    m_tmpl_list = [model_list[0], first_stg_vars_list[0]]
    return model_list, first_stg_vars_list, m_tmpl_list, T


# ------------------------- Persistent wrappers -------------------------
class BaseBundle:
    """每个场景的基础模型（计算真实Q）+ 持久化求解器"""
    def __init__(self, model: pyo.ConcreteModel, options: dict | None = None):
        self.model = model
        self.gp = GurobiPersistent()
        self.gp.set_instance(model)
        if hasattr(model, 'obj'):
            model.del_component('obj')
        model.obj = pyo.Objective(expr=model.obj_expr, sense=pyo.minimize)
        self.gp.set_objective(model.obj)
        if options:
            self.gp.set_gurobi_param('MIPGap', options.get('MIPGap', 1e-1))
            self.gp.set_gurobi_param('NumericFocus', options.get('NumericFocus', 1))
            self.gp.set_gurobi_param('Presolve', options.get('Presolve', 2))
            self.gp.set_gurobi_param('NonConvex', options.get('NonConvex', 2))
            if 'TimeLimit' in options:
                self.gp.set_gurobi_param('TimeLimit', options['TimeLimit'])

    def eval_at(self, first_vars, first_vals):
        for v, val in zip(first_vars, first_vals):
            v.fix(float(val))
            self.gp.update_var(v)
        self.gp.solve(load_solutions=True)
        val = float(pyo.value(self.model.obj_expr))
        for v in first_vars:
            v.unfix()
            self.gp.update_var(v)
        return val

class MSBundle:
    """单场景 ms 子问题（持久化），对一个四面体求解"""
    def __init__(self, model_base: pyo.ConcreteModel, first_vars, options: dict | None = None):
        m = model_base.clone()

        m.lam_index = pyo.RangeSet(0, 3)
        m.lam = pyo.Var(m.lam_index, domain=pyo.NonNegativeReals)
        m.lam_sum = pyo.Constraint(expr=sum(m.lam[j] for j in m.lam_index) == 1.0)

        self.Kp = m.find_component(first_vars[0].name)
        self.Ki = m.find_component(first_vars[1].name)
        self.Kd = m.find_component(first_vars[2].name)
        if any(v is None for v in (self.Kp, self.Ki, self.Kd)):
            raise RuntimeError("克隆模型中找不到 Kp/Ki/Kd")

        m.link_kp = pyo.Constraint(expr=self.Kp == sum(0.0 * m.lam[j] for j in m.lam_index))
        m.link_ki = pyo.Constraint(expr=self.Ki == sum(0.0 * m.lam[j] for j in m.lam_index))
        m.link_kd = pyo.Constraint(expr=self.Kd == sum(0.0 * m.lam[j] for j in m.lam_index))

        m.As = pyo.Var()
        m.As_def = pyo.Constraint(expr=m.As == sum(0.0 * m.lam[j] for j in m.lam_index))

        if hasattr(m, 'obj'):
            m.del_component('obj')
        m.obj = pyo.Objective(expr=m.obj_expr - m.As, sense=pyo.minimize)

        self.model = m
        self.gp = GurobiPersistent()
        self.gp.set_instance(m)
        self.gp.set_objective(m.obj)
        if options:
            self.gp.set_gurobi_param('MIPGap', options.get('MIPGap', 1e-1))
            self.gp.set_gurobi_param('NumericFocus', options.get('NumericFocus', 1))
            self.gp.set_gurobi_param('Presolve', options.get('Presolve', 2))
            self.gp.set_gurobi_param('NonConvex', options.get('NonConvex', 2))
            if 'TimeLimit' in options:
                self.gp.set_gurobi_param('TimeLimit', options['TimeLimit'])

        self.lam = m.lam
        self.link_kp = m.link_kp
        self.link_ki = m.link_ki
        self.link_kd = m.link_kd
        self.As     = m.As
        self.As_def = m.As_def
        self._V_cached = None  # [(x,y,z)]*4

    def _expr_link(self, lhs_var, coeffs):
        return lhs_var == sum(float(coeffs[j]) * self.lam[j] for j in range(4))

    def _replace_constraint(self, attr_name: str, expr):
        old_con = getattr(self.model, attr_name)
        try:
            self.gp.remove_constraint(old_con)
        except Exception:
            pass
        self.model.del_component(old_con)
        new_con = pyo.Constraint(expr=expr)
        self.model.add_component(attr_name, new_con)
        self.gp.add_constraint(getattr(self.model, attr_name))

    def update_tetra(self, tet_vertices, fverts_scene):
        pairs = sorted(
            [(tuple(map(float, tet_vertices[j])), float(fverts_scene[j])) for j in range(4)],
            key=lambda kv: (kv[0][0], kv[0][1], kv[0][2])
        )
        V = [kv[0] for kv in pairs]
        F = [kv[1] for kv in pairs]
        self._V_cached = V

        vx = [V[j][0] for j in range(4)]
        vy = [V[j][1] for j in range(4)]
        vz = [V[j][2] for j in range(4)]

        self._replace_constraint('link_kp', self._expr_link(self.Kp, vx))
        self._replace_constraint('link_ki', self._expr_link(self.Ki, vy))
        self._replace_constraint('link_kd', self._expr_link(self.Kd, vz))
        self._replace_constraint('As_def',  self._expr_link(self.As, F))

    def solve(self):
        res = self.gp.solve(load_solutions=True)
        ok = (res.solver.status == SolverStatus.ok) and \
             (res.solver.termination_condition in {
                 TerminationCondition.optimal,
                 TerminationCondition.locallyOptimal
             })
        return ok

    def get_ms_and_point(self):
        ms_val = float(pyo.value(self.model.obj))
        lam_star = np.array([pyo.value(self.lam[j]) for j in range(4)], dtype=float)
        V = np.array(self._V_cached, dtype=float)
        new_pt = lam_star @ V
        return ms_val, lam_star, tuple(map(float, new_pt))

# ------------------------- Basic utils -------------------------
def corners_from_var_bounds(vars_3):
    bnds = []
    for v in vars_3:
        lb, ub = v.lb, v.ub
        if lb is None or ub is None:
            raise ValueError(f"{v.name} 缺少上下界")
        bnds.append((float(lb), float(ub)))
    return [tuple(p) for p in it.product(*[(lo, hi) for (lo,hi) in bnds])]

def too_close(p, nodes, tol=MIN_DIST):
    return any(np.linalg.norm(np.asarray(p)-np.asarray(q)) < tol for q in nodes)

def evaluate_Q_at(base_bundle: BaseBundle, first_stg_vars, first_stg_vals):
    return base_bundle.eval_at(first_stg_vars, first_stg_vals)

def tet_volume(verts):
    V = np.array(verts, float)
    v0, v1, v2, v3 = V
    return float(abs(np.linalg.det(np.stack([v1 - v0, v2 - v0, v3 - v0], axis=1))) / 6.0)

def tet_quality(verts):
    V = np.array(verts, float)
    edges = [np.linalg.norm(V[i] - V[j]) for (i, j) in it.combinations(range(4), 2)]
    denom = float(np.sum(np.power(edges, 3))) + 1e-16
    vol = tet_volume(verts)
    return float(6.0 * vol / denom)

# ------------------------- Single tetra & scene: ms solve (persistent) -------------------------
def ms_on_tetra_for_scene(ms_bundle: MSBundle, tet_vertices, fverts_scene):
    ms_bundle.update_tetra(tet_vertices, fverts_scene)
    ok = ms_bundle.solve()
    if not ok:
        return float('inf'), None, None
    ms_val, lam_star, new_pt = ms_bundle.get_ms_and_point()
    return ms_val, lam_star, new_pt

# ------------------------- Evaluate all tetrahedra (per-scene) -------------------------
def evaluate_all_tetra(nodes, scen_values, ms_bundles, first_vars_list,
                       ms_cache=None, cache_on=True):
    """
    返回:
      per_tet[k] 包含：
        - ms_per_scene:  长度 S 的 list
        - xms_per_scene: 长度 S 的 list，每个是对应场景的落点 (Kp,Ki,Kd)
        - 兼容字段: ms(聚合), LB, UB, x_ms_best_scene(最优场景落点), best_scene
    说明:
      ms_cache: dict 可选，键为 (scene_idx, tuple(sorted(vert_idx)))，
                值为 (ms_val, new_point)。
      cache_on: 是否启用缓存。
    """
    pts = np.asarray(nodes, dtype=float)
    if len(pts) < 4:
        return None, []
    tri = Delaunay(pts)
    S = len(ms_bundles)

    mins = pts.min(axis=0)
    maxs = pts.max(axis=0)
    diam = float(np.linalg.norm(maxs - mins))
    vol_tol = 1e-12 * max(diam**3, 1.0)

    per_tet = []
    for k, simp in enumerate(tri.simplices):
        idxs = list(map(int, simp))
        verts = [tuple(pts[i]) for i in idxs]

        v0, v1, v2, v3 = np.array(verts)
        vol = abs(np.linalg.det(np.stack([v1 - v0, v2 - v0, v3 - v0], axis=1))) / 6.0
        if vol < vol_tol:
            continue

        # 每个场景在四个顶点上的 f 值
        fverts_per_scene = [[scen_values[ω][i] for i in idxs] for ω in range(S)]
        fverts_sum = [sum(fverts_per_scene[ω][j] for ω in range(S)) for j in range(4)]

        # ========== 带缓存的 per-scene ms 求解 ==========
        key_base = tuple(sorted(idxs))  # 用顶点索引避免浮点坐标键
        ms_scene = []
        xms_scene = []
        for ω in range(S):
            cache_key = (int(ω), key_base)
            hit = (cache_on and (ms_cache is not None) and (cache_key in ms_cache))
            if hit:
                ms_val, new_pt = ms_cache[cache_key]
            else:
                ms_val, lam_star, new_pt = ms_on_tetra_for_scene(
                    ms_bundles[ω], verts, fverts_per_scene[ω]
                )
                if cache_on and (ms_cache is not None):
                    ms_cache[cache_key] = (ms_val, new_pt)
            ms_scene.append(ms_val)
            xms_scene.append(new_pt)
        # ============================================

        if MS_AGG == "sum":
            ms_total = float(np.sum(ms_scene))
        elif MS_AGG == "mean":
            ms_total = float(np.mean(ms_scene))
        else:
            raise ValueError("MS_AGG must be 'sum' or 'mean'")

        LB = float(np.min(fverts_sum) + ms_total)
        UB = float(np.max(fverts_sum) + ms_total)

        best_scene = int(np.argmin(ms_scene))
        x_ms_best = xms_scene[best_scene]

        per_tet.append({
            "simplex_index": k,
            "vert_idx": idxs,
            "verts": verts,
            "fverts_sum": fverts_sum,
            "ms_per_scene": ms_scene,
            "xms_per_scene": xms_scene,
            "ms": ms_total,
            "LB": LB,
            "UB": UB,
            "x_ms_best_scene": x_ms_best,
            "best_scene": best_scene,
            "volume": vol,
        })

    return tri, per_tet


# ------------------------- Pretty print -------------------------
def _print_candidates_table(cands_sorted, nodes, topN=10):
    # 固定每列宽度（你可以按需微调这些数字）
    W = {"rank":4, "simp":6, "scene":7, "ms":12, "mind":12, "pt":30}

    def header_line():
        return (f"{'rank':>{W['rank']}} "
                f"{'simp':>{W['simp']}} "
                f"{'scene':>{W['scene']}} "
                f"{'ms':>{W['ms']}} "
                f"{'mind(all)':>{W['mind']}} "
                f"{'pt':>{W['pt']}}")

    print("== ms candidates (sorted by (ms, -dist)) ==")
    head = header_line()
    print(head)
    print("-" * len(head))

    for rnk, ci in enumerate(cands_sorted[:topN], start=1):
        pt = ci["cand_pt"]
        d  = float('nan') if pt is None else min_dist_to_nodes(pt, nodes)
        simp = f"T{ci['simplex_index']}"
        pt_str = "None" if pt is None else f"({pt[0]:.4f}, {pt[1]:.4f}, {pt[2]:.4f})"
        print(f"{rnk:>{W['rank']}} "
              f"{simp:>{W['simp']}} "
              f"{ci['scene']:>{W['scene']}} "
              f"{ci['cand_ms']:>{W['ms']}.4e} "
              f"{d:>{W['mind']}.2e} "
              f"{pt_str:>{W['pt']}}")

def print_tetra_table(per_tet, active_mask, purple_set=None, prec=6):
    purple_set = set() if purple_set is None else set(purple_set)
    per_tet = sorted(per_tet, key=lambda r: r["simplex_index"])
    tet_ids = [r["simplex_index"] for r in per_tet]
    active_set = {tid for tid in tet_ids if active_mask.get(tid, False)}

    def _mark(tid):
        s = f"T{tid}"
        flags = []
        if tid in active_set:  flags.append("*")
        if tid in purple_set:  flags.append("^")
        return s + ("".join(flags) if flags else "")

    header = ["row\\simp"] + [_mark(tid) for tid in tet_ids]
    rows = [
        ["UB"] + [f"{r['UB']:.{prec}f}" for r in per_tet],
        ["LB"] + [f"{r['LB']:.{prec}f}" for r in per_tet],
        ["ms"] + [f"{r['ms']:.3e}"       for r in per_tet],
    ]
    table = [header] + rows
    colw = [max(len(str(row[c])) for row in table) + 2 for c in range(len(header))]

    RED, PURPLE, RESET = "\033[31m", "\033[35m", "\033[0m"
    def colorize(col_idx, s):
        if col_idx == 0:
            return s
        tid = tet_ids[col_idx-1]
        if tid in purple_set:
            return f"{PURPLE}{s}{RESET}"
        elif tid in active_set:
            return f"{RED}{s}{RESET}"
        return s

    print("\n== Per-tetra summary ==")
    print("".join(colorize(c, str(header[c]).ljust(colw[c])) for c in range(len(header))))
    print("-"*sum(colw))
    for r in rows:
        line = []
        for c in range(len(header)):
            cell = str(r[c])
            pad  = cell.ljust(colw[c]) if c==0 else cell.rjust(colw[c])
            line.append(colorize(c, pad))
        print("".join(line))
    print("(红色列=active；紫色列=包含当前最小节点的单形；第1行=UB，第2行=LB，第3行=ms)\n")

def min_dist_to_nodes(pt, nodes):
    P = np.asarray(pt, float)
    X = np.asarray(nodes, float)
    return float(np.min(np.linalg.norm(X - P, axis=1)))

def print_per_scenario_ms(per_tet, max_scenarios_to_print=10, prec=3):
    per_tet = sorted(per_tet, key=lambda r: r["simplex_index"])
    if not per_tet or "ms_per_scene" not in per_tet[0]:
        return
    S = len(per_tet[0]["ms_per_scene"])
    show = min(S, max_scenarios_to_print)
    head = "simp | " + " ".join([f"s{j}".rjust(10) for j in range(show)])
    print("== Per-tetra per-scenario ms (showing first", show, "of", S, "scenes) ==")
    print(head); print("-"*len(head))
    for r in per_tet:
        arr = r["ms_per_scene"][:show]
        sline = " ".join([f"{v:.{prec}e}".rjust(10) for v in arr])
        print(f"{r['simplex_index']:>4d} | {sline}")
    if show < S:
        print(f"... ({S-show} scenes omitted)")
    print()

# ------------------------- Plotly visualization -------------------------
def plot_iteration_plotly(iter_id, nodes, tri, active_mask, ub_node, next_node, per_tet,
                          highlight_simplices=None):
    import numpy as np
    import plotly.graph_objects as go

    if highlight_simplices is None:
        highlight_simplices = set()
    else:
        highlight_simplices = set(highlight_simplices)

    fig = go.Figure()
    nodes = np.asarray(nodes, float)

    if len(nodes) > 0:
        fig.add_trace(go.Scatter3d(
            x=nodes[:, 0], y=nodes[:, 1], z=nodes[:, 2],
            mode='markers',
            marker=dict(size=4, color="black"),
            name='nodes'
        ))

    if ub_node is not None:
        fig.add_trace(go.Scatter3d(
            x=[ub_node[0]], y=[ub_node[1]], z=[ub_node[2]],
            mode='markers',
            marker=dict(size=7, symbol="circle", color="green"),
            name='current min node'
        ))

    if next_node is not None:
        fig.add_trace(go.Scatter3d(
            x=[next_node[0]], y=[next_node[1]], z=[next_node[2]],
            mode='markers',
            marker=dict(size=8, symbol="diamond", color="#1976d2"),
            name='next node'
        ))

    def _is_same_point(a, b, atol=1e-6):
        if a is None or b is None:
            return False
        return np.linalg.norm(np.asarray(a, float) - np.asarray(b, float)) <= float(atol)

    if tri is not None:
        legend_mesh_added = False
        legend_edge_added = False

        for r in per_tet:
            sid = r["simplex_index"]
            if not active_mask.get(sid, False):
                continue

            verts = np.array(r["verts"], dtype=float)

            # 允许 next_node 与该单形的任一场景落点重合时高亮
            highlight_by_next = _is_same_point(next_node, r.get("x_ms_best_scene", None), atol=1e-6)
            if (not highlight_by_next) and ("xms_per_scene" in r):
                for pt_s in r["xms_per_scene"]:
                    if _is_same_point(next_node, pt_s, atol=1e-6):
                        highlight_by_next = True
                        break

            highlight = highlight_by_next or (sid in highlight_simplices)

            mesh_color = "#ff5722" if highlight else "#ffb74d"
            edge_color = "darkorange"
            edge_width = 4 if highlight else 3
            mesh_opacity = 0.45 if highlight else 0.35

            I = [0, 0, 0, 1]
            J = [1, 1, 2, 2]
            K = [2, 3, 3, 3]

            fig.add_trace(go.Mesh3d(
                x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
                i=I, j=J, k=K,
                color=mesh_color,
                opacity=mesh_opacity,
                showscale=False,
                name="active simplex",
                showlegend=(not legend_mesh_added)
            ))
            legend_mesh_added = True

            edges = [(0,1), (0,2), (0,3), (1,2), (1,3), (2,3)]
            for (a, b) in edges:
                pa, pb = verts[a], verts[b]
                fig.add_trace(go.Scatter3d(
                    x=[pa[0], pb[0]],
                    y=[pa[1], pb[1]],
                    z=[pa[2], pb[2]],
                    mode='lines',
                    line=dict(width=edge_width, color=edge_color),
                    name='active edge',
                    showlegend=(not legend_edge_added)
                ))
            legend_edge_added = True

            cx, cy, cz = np.mean(verts, axis=0)
            qtxt = ""
            if "quality" in r and r["quality"] is not None:
                try:
                    qtxt = f"<br>q={float(r['quality']):.3e}"
                except Exception:
                    qtxt = ""
            txt = (f"simp={sid}"
                   f"<br>LB={float(r['LB']):.6f}"
                   f"<br>UB={float(r['UB']):.6f}"
                   f"<br>ms={float(r['ms']):.3e}"
                   f"<br>vol={float(r['volume']):.3e}"
                   f"{qtxt}")

            fig.add_trace(go.Scatter3d(
                x=[cx], y=[cy], z=[cz],
                mode='markers',
                marker=dict(size=1, opacity=0.0),
                text=[txt], hoverinfo="text",
                name="tetra info",
                showlegend=False
            ))

    fig.update_layout(
        title=f"Iteration {iter_id}",
        scene=dict(
            xaxis_title="Kp",
            yaxis_title="Ki",
            zaxis_title="Kd",
            aspectmode="cube",
            zaxis=dict(tickformat=".2f"),
        ),
        width=980,
        height=720,
        legend=dict(itemsizing="constant")
    )
    fig.update_traces(
        hovertemplate="x: %{x:.6f}<br>y: %{y:.6f}<br>z: %{z:.6f}",
        selector=dict(type='scatter3d')
    )
    fig.show()

# ------------------------- MAIN LOOP -------------------------
def run_pid_simplex_3d(base_bundles, ms_bundles, model_list, first_vars_list,
                       target_nodes=30, min_dist=MIN_DIST, active_tol=ACTIVE_TOL, verbose=True,
                       agg_bundle=None):
    """
    本次实现：只使用“单场景 ms”（即 ms_bundles），且**仅在包含 UB 节点的 active 单形**中，
    对每个场景分别产生候选点，从所有(单形×场景)候选中选 ms 最小者作为 next node。
    """
    global LAST_DEBUG
    LB_hist, UB_hist, ms_hist, node_count = [], [], [], []
    UB_node_hist, add_node_hist = [], []
    ms_a_hist, ms_b_hist = [], []
    active_ratio_hist = []

    S = len(model_list)
    nodes = corners_from_var_bounds(first_vars_list[0])

    bounds_arr = np.array([[float(v.lb), float(v.ub)] for v in first_vars_list[0]], float)
    diam = float(np.linalg.norm(bounds_arr[:,1] - bounds_arr[:,0]))
    min_dist = float(min_dist)

    # 缓存 f_ω(node_i)
    scen_values = [[None]*len(nodes) for _ in range(S)]
    for i, node in enumerate(nodes):
        for ω in range(S):
            scen_values[ω][i] = evaluate_Q_at(base_bundles[ω], first_vars_list[ω], node)

    it = 0
    stop_due_to_collision = False
    ms_cache = {}   # <== 新增： (scene_idx, sorted(vert_idx)) -> (ms, cand_pt)
    while len(nodes) < target_nodes:
        # 1) 全局 UB（按 sum 目标）
        f_sum_per_node = [
            sum(scen_values[ω][i] for ω in range(S))
            for i in range(len(nodes))
        ]
        ub_idx = int(np.argmin(f_sum_per_node))
        UB_global = float(f_sum_per_node[ub_idx])
        UB_node = tuple(nodes[ub_idx])

        # 2) 评估所有四面体（单场景 ms）
        tri, per_tet = evaluate_all_tetra(
            nodes, scen_values, ms_bundles, first_vars_list,
            ms_cache=ms_cache, cache_on=MS_CACHE_ENABLE
        )

        if tri is None or not per_tet:
            if verbose:
                print("Not enough nodes to make tetrahedra; stop.")
            break

        # 3) active mask（按 UB 过滤 + 形状质量）
        active_mask = {
            r["simplex_index"]: (r["LB"] <= UB_global + active_tol)
            for r in per_tet
        }
        q_cut = 1e-3
        for r in per_tet:
            sid = r["simplex_index"]
            if not active_mask.get(sid, False):
                continue
            q = tet_quality(r["verts"])
            if q < q_cut:
                active_mask[sid] = False

        # 4) active ratio
        total_vol = sum(r["volume"] for r in per_tet)
        active_vol = sum(r["volume"] for r in per_tet if active_mask[r["simplex_index"]])
        active_ratio = active_vol / total_vol if total_vol > 0 else 0.0

        # 5) LB_global & ms_b
        ub_active = [r for r in per_tet
                     if (ub_idx in r["vert_idx"]) and active_mask.get(r["simplex_index"], False)]
        if ub_active:
            ms_b_rec   = min(ub_active, key=lambda r: r["ms"])
            ms_b       = float(ms_b_rec["ms"])
            ms_b_simp  = int(ms_b_rec["simplex_index"])
            LB_global  = UB_global + ms_b
        else:
            ms_b       = float('nan')
            ms_b_simp  = None
            active_LBs = [r["LB"] for r in per_tet if active_mask.get(r["simplex_index"], False)]
            LB_global  = float(min(active_LBs)) if active_LBs else float(min(r["LB"] for r in per_tet))

        # 6) ms_a（active 内最小聚合 ms，作为历史记录保持）
        if any(active_mask.values()):
            ms_a = float(min(r["ms"] for r in per_tet if active_mask[r["simplex_index"]]))
        else:
            ms_a = float(min(r["ms"] for r in per_tet))
        ms_iter = ms_a

        # 7) 记录
        LB_hist.append(LB_global)
        UB_hist.append(UB_global)
        ms_hist.append(ms_iter)
        node_count.append(len(nodes))
        UB_node_hist.append(UB_node)
        ms_a_hist.append(ms_a)
        ms_b_hist.append(ms_b)
        active_ratio_hist.append(active_ratio)

        # 8) 打印
        simp_with_min = [r["simplex_index"] for r in per_tet if ub_idx in r["vert_idx"]]
        if verbose:
            print(f"[Iter {it}] Active simplex ratio = {active_ratio:.6f}")
            print(f"[Iter {it}] UB node {UB_node} is in simplices {sorted(simp_with_min)}")
            msb_src = f"T{ms_b_simp}" if ms_b_simp is not None else "N/A"
            print(f"[Iter {it}] LB = {LB_global:.6f} = UB({UB_global:.6f}) + ms_b({ms_b:.3e}) from {msb_src}")

        # 9) 候选排行（仅 UB 邻域的 active 单形 × 所有场景）
        active = [r for r in per_tet if active_mask[r["simplex_index"]]]
        ub_active = [r for r in active if ub_idx in r["vert_idx"]]
        pool_records = ub_active if len(ub_active) > 0 else active

        # 构造成“项=单形×场景”的候选
        cand_items = []
        for rec in pool_records:
            sid = rec["simplex_index"]
            ms_list = rec.get("ms_per_scene", [])
            pts_list = rec.get("xms_per_scene", [None]*len(ms_list))
            for s in range(len(ms_list)):
                cand_items.append({
                    "simplex_index": sid,
                    "scene": s,
                    "cand_ms": ms_list[s],
                    "cand_pt": pts_list[s],
                    # 附带原记录用于调试
                    "_rec": rec
                })

        def score_item(ci):
            ms = ci["cand_ms"]
            pt = ci["cand_pt"]
            d  = (float('inf') if pt is None else min_dist_to_nodes(pt, nodes))
            return (ms, -d)

        candidates_sorted = sorted(cand_items, key=score_item)

        if verbose:
            top_msg = "N/A"
            if len(candidates_sorted) > 0:
                t0 = candidates_sorted[0]
                top_msg = f"T{int(t0['simplex_index'])}, scene={t0['scene']}, ms={float(t0['cand_ms']):.3e}"
            msb_src = f"T{ms_b_simp}" if ms_b_simp is not None else "N/A"
            print(f"[Iter {it}] LB = {LB_global:.6f} = UB({UB_global:.6f}) + ms_b({ms_b:.3e}) from {msb_src}")
            print(f"[Iter {it}] candidate rank #1: {top_msg}")

            _print_candidates_table(candidates_sorted, nodes, topN=10)
            print()

        # 10) 选新点 + 强校验/撞车处理
        new_node = None
        chosen_ms = None
        chosen_cand = None
        stop_due_to_collision = False

        def handle_collision(cand_pt, ci, stage_note="active"):
            nonlocal stop_due_to_collision
            X = np.asarray(nodes, float)
            P = np.asarray(cand_pt, float)
            dists = np.linalg.norm(X - P, axis=1)
            j_star = int(np.argmin(dists))
            d_star = float(dists[j_star])
            orange_ids = [r["simplex_index"] for r in per_tet if j_star in r["vert_idx"]]
            debug_pack = {
                "reason": "candidate_too_close",
                "iter": it,
                "stage": stage_note,
                "min_dist": float(min_dist),
                "closest_node_index": j_star,
                "closest_node_point": tuple(map(float, nodes[j_star])),
                "closest_distance": d_star,
                "cand_simplex": int(ci["simplex_index"]),
                "cand_scene": int(ci["scene"]),
                "cand_point": tuple(map(float, cand_pt)),
                "cand_ms": float(ci["cand_ms"]),
                "UB_global": float(UB_global),
                "LB_global": float(LB_global),
                "active_ratio": float(active_ratio),
                "UB_node": tuple(map(float, UB_node)),
                "active_mask": {int(k): bool(v) for k, v in active_mask.items()},
                "nodes_snapshot": [tuple(map(float, nd)) for nd in nodes],
                "per_tet_snapshot": [
                    {
                        "simplex_index": int(r["simplex_index"]),
                        "vert_idx": list(map(int, r["vert_idx"])),
                        "verts": [tuple(map(float, x)) for x in r['verts']],
                        "ms": float(r["ms"]),
                        "ms_per_scene": [float(x) for x in r.get("ms_per_scene", [])],
                        "LB": float(r["LB"]),
                        "UB": float(r["UB"]),
                        "best_scene": int(r["best_scene"]),
                        "x_ms_best_scene": tuple(map(float, r["x_ms_best_scene"])) if r.get("x_ms_best_scene") is not None else None,
                        "volume": float(r["volume"]),
                    } for r in per_tet
                ],
                "highlight_simplices": list(map(int, orange_ids)),
            }
            global LAST_DEBUG
            LAST_DEBUG = debug_pack
            plot_iteration_plotly(
                it, nodes, tri, active_mask, UB_node, cand_pt, per_tet,
                highlight_simplices=orange_ids
            )
            if verbose:
                print(
                    f"[STOP] Candidate {tuple(map(float, cand_pt))} "
                    f"(scene {ci['scene']}) is too close to existing node #{j_star} at distance {d_star:.3e} "
                    f"(< {min_dist:g}). Highlighted simplices: {sorted(orange_ids)}"
                )
            stop_due_to_collision = True

        for rank, ci in enumerate(candidates_sorted, start=1):
            cand_pt = ci["cand_pt"]
            if cand_pt is None:
                continue
            if min_dist_to_nodes(cand_pt, nodes) >= min_dist:
                new_node   = cand_pt
                chosen_ms  = ci["cand_ms"]
                chosen_cand= ci
                if verbose:
                    print(
                        f"Chosen node {tuple(map(float, cand_pt))} "
                        f"with ms={chosen_ms:.3e} "
                        f"(simp T{ci['simplex_index']}, scene {ci['scene']}, rank #{rank})"
                    )
                    print(f"[Iter {it}] next node comes from simplex T{int(ci['simplex_index'])}, scene {int(ci['scene'])}")
                break
            else:
                if verbose:
                    print(
                        f"Skip candidate {tuple(map(float, cand_pt))} "
                        f"(simp T{ci['simplex_index']}, scene {ci['scene']}, rank #{rank}) "
                        f"because too close to existing nodes (< {min_dist:g})."
                    )
                handle_collision(cand_pt, ci, stage_note="active")
                break

        if (new_node is None) and (not stop_due_to_collision) and (len(active) > 0):
            if verbose:
                print("[fallback] All UB-neighborhood candidates too close; try all active simplices × scenes...")
            # 放宽到所有 active × scene
            cand_items_all = []
            for rec in active:
                sid = rec["simplex_index"]
                ms_list = rec.get("ms_per_scene", [])
                pts_list = rec.get("xms_per_scene", [None]*len(ms_list))
                for s in range(len(ms_list)):
                    cand_items_all.append({
                        "simplex_index": sid,
                        "scene": s,
                        "cand_ms": ms_list[s],
                        "cand_pt": pts_list[s],
                        "_rec": rec
                    })
            for ci in sorted(cand_items_all, key=score_item):
                cand_pt = ci["cand_pt"]
                if cand_pt is None: 
                    continue
                if min_dist_to_nodes(cand_pt, nodes) >= min_dist:
                    new_node   = cand_pt
                    chosen_ms  = ci["cand_ms"]
                    chosen_cand= ci
                    if verbose:
                        print(
                            f"Chosen node {tuple(map(float, cand_pt))} "
                            f"with ms={chosen_ms:.3e} "
                            f"(simp T{ci['simplex_index']}, scene {ci['scene']}) [fallback-active]"
                        )
                    break
                else:
                    if verbose:
                        print(
                            f"Skip (active) candidate {tuple(map(float, cand_pt))} "
                            f"(simp T{ci['simplex_index']}, scene {ci['scene']}) "
                            f"because too close to existing nodes (< {min_dist:g})."
                        )
                    handle_collision(cand_pt, ci, stage_note="fallback-active")
                    break

        if stop_due_to_collision:
            if verbose:
                print(f"[Iter {it}] Stop due to collision.")
            break

        if new_node is None:
            if verbose:
                print("New node too close for all candidates (or infeasible ms); stop.")
            break

        # == 强校验：避免 next_node 等于顶点 ==
        tol_same = 1e-10
        def _same(a, b, tol=tol_same):
            a = np.asarray(a, float); b = np.asarray(b, float)
            return np.linalg.norm(a - b) <= tol

        if chosen_cand is not None and "_rec" in chosen_cand:
            rec = chosen_cand["_rec"]
            offending_vert = None
            for v in rec["verts"]:
                if _same(new_node, v):
                    offending_vert = tuple(map(float, v))
                    break
            if offending_vert is not None:
                LAST_DEBUG = {
                    "reason": "next_node_equals_vertex",
                    "iter": it,
                    "new_node": tuple(map(float, new_node)),
                    "offending_vertex": offending_vert,
                    "tol_same": tol_same,
                    "candidate": {
                        "simplex_index": int(rec["simplex_index"]),
                        "scene": int(chosen_cand["scene"]),
                        "vert_idx": list(map(int, rec["vert_idx"])),
                        "verts": [tuple(map(float, x)) for x in rec["verts"]],
                        "ms": float(chosen_cand["cand_ms"]),
                        "ms_per_scene": [float(x) for x in rec["ms_per_scene"]],
                        "best_scene": int(rec["best_scene"]),
                        "x_ms_best_scene": tuple(map(float, rec["x_ms_best_scene"])) if rec.get("x_ms_best_scene") is not None else None,
                        "LB": float(rec["LB"]),
                        "UB": float(rec["UB"]),
                        "volume": float(rec["volume"]),
                    },
                    "UB_global": float(UB_global),
                    "LB_global": float(LB_global),
                    "active_ratio": float(active_ratio),
                    "UB_node": tuple(map(float, UB_node)),
                    "active_mask": {int(k): bool(v) for k, v in active_mask.items()},
                    "nodes_snapshot": [tuple(map(float, nd)) for nd in nodes],
                    "per_tet_snapshot": [
                        {
                            "simplex_index": int(r["simplex_index"]),
                            "vert_idx": list(map(int, r["vert_idx"])),
                            "verts": [tuple(map(float, x)) for x in r["verts"]],
                            "ms": float(r["ms"]),
                            "ms_per_scene": [float(x) for x in r.get("ms_per_scene", [])],
                            "LB": float(r["LB"]),
                            "UB": float(r["UB"]),
                            "best_scene": int(r["best_scene"]),
                            "x_ms_best_scene": tuple(map(float, r["x_ms_best_scene"])) if r.get("x_ms_best_scene") is not None else None,
                            "volume": float(r["volume"]),
                        } for r in per_tet
                    ],
                }
                # 高亮与 offending 顶点相邻的单形
                vert_idx_list = []
                for j, nd in enumerate(nodes):
                    if _same(offending_vert, nd):
                        vert_idx_list.append(j)
                orange_ids = [r["simplex_index"] for r in per_tet if any(j in r["vert_idx"] for j in vert_idx_list)]
                plot_iteration_plotly(it, nodes, tri, active_mask, UB_node, new_node, per_tet,
                                      highlight_simplices=orange_ids)
                if verbose:
                    print("[STOP] new_node coincides with a simplex vertex. Highlighted simplices:",
                          sorted(orange_ids))
                break

        # 可视化（正常迭代）
        plot_iteration_plotly(it, nodes, tri, active_mask, UB_node, new_node, per_tet,
                              highlight_simplices=None)

        # 加点并评估（持久化 base）
        new_vals = []
        for ω in range(S):
            val = evaluate_Q_at(base_bundles[ω], first_vars_list[ω], new_node)
            new_vals.append(val)

        nodes.append(tuple(map(float, new_node)))
        for ω in range(S):
            scen_values[ω].append(new_vals[ω])

        add_node_hist.append(new_node)
        it += 1

    return {
        "nodes": np.array(nodes, float),
        "LB_hist": LB_hist,
        "UB_hist": UB_hist,
        "ms_hist": ms_hist,
        "ms_a_hist": ms_a_hist,
        "ms_b_hist": ms_b_hist,
        "node_count": node_count,
        "UB_node_hist": UB_node_hist,
        "added_nodes": add_node_hist,
        "active_ratio_hist": active_ratio_hist,
    }

# ===================== MAIN =====================
RUN_QUICK_TEST = True  # True: 先用小规模验证

if RUN_QUICK_TEST:
    csv_path       = "data.csv"
    max_scenarios  = 1
    target_nodes   = 13
else:
    csv_path       = "data.csv"
    max_scenarios  = 99
    target_nodes   = 30

bounds = {
    "x":  (None, None),
    "u":  (None, None),
    "e":  (None, None),
    "I":  (-10, 10),
    "Kp": (0, 1),
    "Ki": (0, 1),
    "Kd": (0, 1),
}
weights = (1.0, 0.01)

# build models
model_list, first_stg_vars_list, m_tmpl_list, T = build_models_from_csv(
    csv_path, h=0.2, weights=weights, bounds=bounds,
    sp0=0.0, sp1=0.5, ku_col="tau_us", tau_col="tau_xs",
    disturb_prefix="disturbance_", setpoint_change_col="setpoint_change",
    max_scenarios=max_scenarios, skip=0
)

# gurobi 参数
gurobi_options = {
    'MIPGap': 1e-1,
    'NumericFocus': 1,
    'Presolve': 2,
    'NonConvex': 2,   # 必须
    'TimeLimit': 10,  # 可按需打开/删除
}

# 持久化封装（基础 + 单场景 ms）
base_bundles = [BaseBundle(m, gurobi_options) for m in model_list]
ms_bundles   = [MSBundle(m, yvars, gurobi_options) for m, yvars in zip(model_list, first_stg_vars_list)]

# 你的需求是“对每个场景分别算”，因此这里不启用共享 λ 聚合
agg_bundle = None

# run
hist = run_pid_simplex_3d(
    base_bundles=base_bundles,
    ms_bundles=ms_bundles,
    model_list=model_list,
    first_vars_list=first_stg_vars_list,
    target_nodes=target_nodes,
    min_dist=MIN_DIST,
    active_tol=ACTIVE_TOL,
    verbose=True,
    agg_bundle=agg_bundle
)

print("\n==== Done ====")
print(f"Total nodes: {len(hist['nodes'])}")
print(f"Best UB: {min(hist['UB_hist']) if hist['UB_hist'] else None}")
print(f"Last LB: {hist['LB_hist'][-1] if hist['LB_hist'] else None}")
# ================================================================================================================


Set parameter MIPGap to value 0.1
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter TimeLimit to value 10
Set parameter MIPGap to value 0.1
Set parameter NumericFocus to value 1
Set parameter Presolve to value 2
Set parameter NonConvex to value 2
Set parameter TimeLimit to value 10
[Iter 0] Active simplex ratio = 1.000000
[Iter 0] UB node (1.0, 1.0, 1.0) is in simplices [2, 3]
[Iter 0] LB = 0.162061 = UB(0.361626) + ms_b(-1.996e-01) from T2
[Iter 0] LB = 0.162061 = UB(0.361626) + ms_b(-1.996e-01) from T2
[Iter 0] candidate rank #1: T2, scene=0, ms=-1.996e-01
== ms candidates (sorted by (ms, -dist)) ==
rank   simp   scene           ms    mind(all)                             pt
----------------------------------------------------------------------------
   1     T2       0  -1.9957e-01     5.23e-01       (1.0000, 0.3698, 0.3698)
   2     T3       0  -1.9957e-01     5.23e-01       (1.0000, 0.3698, 0.3698)

Chosen node

[Iter 1] Active simplex ratio = 1.000000
[Iter 1] UB node (1.0, 1.0, 1.0) is in simplices [2, 3, 8, 9, 10, 11]
[Iter 1] LB = -0.558282 = UB(0.361626) + ms_b(-9.199e-01) from T2
[Iter 1] LB = -0.558282 = UB(0.361626) + ms_b(-9.199e-01) from T2
[Iter 1] candidate rank #1: T2, scene=0, ms=-9.199e-01
== ms candidates (sorted by (ms, -dist)) ==
rank   simp   scene           ms    mind(all)                             pt
----------------------------------------------------------------------------
   1     T2       0  -9.1991e-01     4.41e-01       (0.3117, 0.3117, 1.0000)
   2     T3       0  -9.1991e-01     4.41e-01       (0.3117, 0.3117, 1.0000)
   3    T11       0  -2.8997e-01     4.54e-01       (0.3214, 1.0000, 0.3214)
   4    T10       0  -2.8997e-01     4.54e-01       (0.3214, 1.0000, 0.3214)

Chosen node (0.3116794718137422, 0.31167946840807353, 0.9999999984121736) with ms=-9.199e-01 (simp T2, scene 0, rank #1)
[Iter 1] next node comes from simplex T2, scene 0


[Iter 2] Active simplex ratio = 0.905630
[Iter 2] UB node (1.0, 1.0, 1.0) is in simplices [4, 5, 8, 9, 10, 11, 16, 17]
[Iter 2] LB = 0.071659 = UB(0.361626) + ms_b(-2.900e-01) from T11
[Iter 2] LB = 0.071659 = UB(0.361626) + ms_b(-2.900e-01) from T11
[Iter 2] candidate rank #1: T11, scene=0, ms=-2.900e-01
== ms candidates (sorted by (ms, -dist)) ==
rank   simp   scene           ms    mind(all)                             pt
----------------------------------------------------------------------------
   1    T11       0  -2.8997e-01     4.54e-01       (0.3214, 1.0000, 0.3214)
   2    T10       0  -2.8997e-01     4.54e-01       (0.3214, 1.0000, 0.3214)
   3     T5       0  -1.3590e-01     3.67e-01       (0.5709, 0.5709, 1.0000)
   4     T4       0  -1.3590e-01     3.67e-01       (0.5709, 0.5709, 1.0000)

Chosen node (0.3213541091906367, 0.999999993093919, 0.3213541134696063) with ms=-2.900e-01 (simp T11, scene 0, rank #1)
[Iter 2] next node comes from simplex T11, scene 0


[Iter 3] Active simplex ratio = 0.863803
[Iter 3] UB node (1.0, 1.0, 1.0) is in simplices [2, 3, 4, 5, 11, 12, 17, 18, 21, 22]
[Iter 3] LB = 0.222111 = UB(0.361626) + ms_b(-1.395e-01) from T4
[Iter 3] LB = 0.222111 = UB(0.361626) + ms_b(-1.395e-01) from T4
[Iter 3] candidate rank #1: T4, scene=0, ms=-1.395e-01
== ms candidates (sorted by (ms, -dist)) ==
rank   simp   scene           ms    mind(all)                             pt
----------------------------------------------------------------------------
   1     T4       0  -1.3952e-01     3.72e-01       (0.4631, 0.6153, 0.8478)
   2     T5       0  -1.3952e-01     3.72e-01       (0.4631, 0.6153, 0.8478)
   3     T2       0  -1.3590e-01     3.67e-01       (0.5709, 0.5709, 1.0000)
   4     T3       0  -5.7121e-02     4.20e-01       (1.0000, 0.8296, 0.3836)

Chosen node (0.4631080022651556, 0.615295912439594, 0.847812104697146) with ms=-1.395e-01 (simp T4, scene 0, rank #1)
[Iter 3] next node comes from simplex T4, scene 0


[Iter 4] Active simplex ratio = 0.826469
[Iter 4] UB node (1.0, 1.0, 1.0) is in simplices [0, 1, 2, 7, 9, 10, 21, 22, 25, 26, 27, 28]
[Iter 4] LB = 0.225722 = UB(0.361626) + ms_b(-1.359e-01) from T0
[Iter 4] LB = 0.225722 = UB(0.361626) + ms_b(-1.359e-01) from T0
[Iter 4] candidate rank #1: T0, scene=0, ms=-1.359e-01
== ms candidates (sorted by (ms, -dist)) ==
rank   simp   scene           ms    mind(all)                             pt
----------------------------------------------------------------------------
   1     T0       0  -1.3590e-01     1.92e-01       (0.5709, 0.5709, 1.0000)
   2     T1       0  -1.3590e-01     1.92e-01       (0.5709, 0.5709, 1.0000)
   3     T7       0  -1.1924e-01     4.02e-01       (1.0000, 0.4023, 1.0000)
   4     T9       0  -5.7121e-02     4.20e-01       (1.0000, 0.8296, 0.3836)
   5    T10       0  -5.0312e-02     4.47e-01       (0.6991, 0.8481, 0.5473)
   6     T2       0  -4.8200e-02     3.85e-01       (0.5506, 0.9283, 0.6223)

Chosen node (0.57091


==== Done ====
Total nodes: 13
Best UB: 0.3616264563470583
Last LB: 0.22572154739802636
